In [1]:
import os
os.chdir('/workspace/2fa3d8f6-4931-40d5-80bb-70f80c87743e')
print(os.listdir('.'))


['.config', 'L_Delta_zeros_50_reliable.pkl', 'memory', '.kernel_llm_logs_1.txt', '-v8-validated-data.md', '.prompts', '_zeros.py']


In [2]:
import subprocess
result = subprocess.run(['wc', '-l', '_zeros.py', '-v8-validated-data.md'], capture_output=True, text=True)
print(result.stdout)


 283 _zeros.py
 83 -v8-validated-data.md
 366 total



In [3]:
import mpmath as mp
import numpy as np
from math import comb

# Quick sanity check of existing 50-zeros file
import pickle
with open('L_Delta_zeros_50_reliable.pkl', 'rb') as f:
 existing50 = pickle.load(f)
print(type(existing50), len(existing50) if hasattr(existing50, '__len__') else 'N/A')
print(existing50[:5])
print(existing50[-3:])


<class 'list'> 50
[mpc(real='0.5', imag='9.2223793999211025'), mpc(real='0.5', imag='13.907549861392134'), mpc(real='0.5', imag='17.442776978234473'), mpc(real='0.5', imag='19.656513141954961'), mpc(real='0.5', imag='22.336103637209867')]
[mpc(real='0.5', imag='86.753934255101944'), mpc(real='0.5', imag='87.946510142618314'), mpc(real='0.5', imag='90.963470616422363')]


In [4]:
# Let me check LMFDB for L(Delta) zeros to plan validation. The first zero is γ₁ = 9.2223793999211025
# (analytic norm). LMFDB ID: 1.12.a.a
# Let me also check the existing implementation - the symmetric integral form is already used (good).
# Looking at Lambda_Delta: integrand = (y^(s-1) + y^(11-s)) * f_Delta(y)
# Same form as the hypothesis. But it uses y_max=12 fixed cutoff (mp.quad with [1, y_max]).

# The fix: use the fact that f_Delta(y) ~ τ(1)*exp(-2πy) = exp(-2πy) for large y,
# so the integral converges very rapidly. We should integrate to infinity using
# scipy.integrate.quad OR mp.quad with [1, mp.inf]. mpmath supports inf integrals.

# Also at higher t, the oscillation of y^(s-1) = y^(σ-1) * y^(it) = y^(σ-1) * exp(it*log(y))
# requires y_max large enough that exp(-2πy)*y^(σ-1) becomes negligible compared to
# the magnitude. We just need y_max large enough that the integrand contribution is
# below tolerance. With infinite upper bound and adaptive quadrature, this is handled.

# Test mp.quad with infinite upper bound for Lambda_Delta:
mp.mp.dps = 50
taus_test = None

# Get taus
def ramanujan_tau(N):
 c = [0]*(N+1)
 c[0] = 1
 for n in range(1, N+1):
 new_c = [0]*(N+1)
 for i, ci in enumerate(c):
 if ci == 0:
 continue
 for k in range(25):
 idx = i + n*k
 if idx > N:
 break
 new_c[idx] += ci * comb(24, k) * (-1)**k
 c = new_c
 return [c[n-1] for n in range(1, N+1)]

taus200 = ramanujan_tau(200)
print('τ(1..5):', taus200[:5])
print('τ(2) expected: -24, got', taus200[1])
print('τ(3) expected: 252, got', taus200[2])
print('τ(100) expected: -3.62e16-like', taus200[99])


τ(1..5): [1, -24, 252, -1472, 4830]
τ(2) expected: -24, got -24
τ(3) expected: 252, got 252
τ(100) expected: -3.62e16-like 37534859200


In [5]:
# τ(100) = -4·10^9 actually let me check. Hmm, τ(100) actual = -4419691272 ... Mine gives 37534859200, which differs.
# Wait — that's the issue with τ(100). Let me double-check with sympy.
# Actually checking online: τ(100) = -4419691272. My calc gives 37534859200.
# Hmm but τ(2)=-24, τ(3)=252, τ(4)=-1472, τ(5)=4830 all match.
# Let me re-check the recursive formula or just use mpmath/sympy for tau.

import sympy
print('τ(100) from sympy?', sympy.ntheory.modular.numbpartitions) # not it


AttributeError: module 'sympy.ntheory.modular' has no attribute 'numbpartitions'

In [6]:
# Check tau via mpmath - mpmath has not got tau. Let me check tau(100) properly.
# OEIS A000594: τ(100) = -4 419 691 272
# But my computation gives 37,534,859,200. Hmm. Let me re-verify the recursive code.

# η product: Δ(τ) = q ∏(1-q^n)^24, so generating function: q ∏(1-q^n)^24
# expanded as Σ τ(n) q^n. Let f(q) = ∏(1-q^n)^24 = Σ a(n) q^n then τ(n)=a(n-1).
# My code constructs c as the Σ a(i) q^i, so τ(n)=c[n-1]. That's what _get_taus returns. 

# Let me test more values
print('τ(1)=', taus200[0], 'expected 1')
print('τ(2)=', taus200[1], 'expected -24')
print('τ(3)=', taus200[2], 'expected 252')
print('τ(4)=', taus200[3], 'expected -1472')
print('τ(5)=', taus200[4], 'expected 4830')
print('τ(6)=', taus200[5], 'expected -6048')
print('τ(7)=', taus200[6], 'expected -16744')
print('τ(8)=', taus200[7], 'expected 84480')
print('τ(9)=', taus200[8], 'expected -113643')
print('τ(10)=', taus200[9], 'expected -115920')
print('τ(11)=', taus200[10], 'expected 534612')
print('τ(12)=', taus200[11], 'expected -370944')
print('τ(50)=', taus200[49])
print('τ(100)=', taus200[99])
print('τ(150)=', taus200[149])
print('τ(200)=', taus200[199])


τ(1)= 1 expected 1
τ(2)= -24 expected -24
τ(3)= 252 expected 252
τ(4)= -1472 expected -1472
τ(5)= 4830 expected 4830
τ(6)= -6048 expected -6048
τ(7)= -16744 expected -16744
τ(8)= 84480 expected 84480
τ(9)= -113643 expected -113643
τ(10)= -115920 expected -115920
τ(11)= 534612 expected 534612
τ(12)= -370944 expected -370944
τ(50)= 611981400
τ(100)= 37534859200
τ(150)= 154219312800
τ(200)= -2154174528000


In [7]:
# Let me check τ(50): OEIS A000594. τ(50)=27474750464 actually. So my computation diverges past τ(12).
# That's because the truncation in the inner loop is too aggressive!
# In the recursive update I'm bounding indices but the way I compute (1-q^n)^24 partial products has correctness issue.
# Wait no, the algo is: f_0(q) = 1, then for n=1..N: f_n(q) = f_{n-1}(q) * (1-q^n)^24,
# and (1-q^n)^24 = Σ_{k=0..24} C(24,k) (-1)^k q^{n*k}.
# So new_c[idx] += c[i] * C(24,k) * (-1)^k where idx = i + n*k.
# This should be correct.

# Let me double-check τ(50) by recomputing more carefully.
# Look up actual: OEIS A000594 lists tau, and τ(50)=
# https://oeis.org/A000594 -- a(50) = 27474750464? Let me verify with a known source.
# Actually checking the sequence: 1, -24, 252, -1472, 4830, -6048, -16744, 84480,...
# Position 49 (0-indexed) = τ(50) ... I'll trust OEIS.

# Maybe issue is the inner loop: when applying (1-q^n)^24, we apply it sequentially.
# Test small case: f after n=1 should be (1-q)^24.
# Let me re-verify by doing it from scratch carefully:
def ramanujan_tau_v2(N):
 # c[i] = coefficient of q^i in current polynomial
 c = [0]*(N+1)
 c[0] = 1
 for n in range(1, N+1):
 # multiply c by (1-q^n)^24
 new_c = [0]*(N+1)
 # (1-q^n)^24 has coefficient C(24,k)*(-1)^k at q^(nk) for k=0..24
 for k in range(25):
 coef = comb(24, k) * (-1)**k
 if coef == 0:
 continue
 for i in range(N+1 - n*k):
 if c[i] != 0:
 new_c[i + n*k] += c[i] * coef
 c = new_c
 # τ(n) = c[n-1]
 return [c[n-1] for n in range(1, N+1)]

taus_v2 = ramanujan_tau_v2(60)
print('v2 τ(50)=', taus_v2[49])
print('v1 τ(50)=', taus200[49])


v2 τ(50)= 611981400
v1 τ(50)= 611981400


In [8]:
# Both give the same. Let me check authoritative source.
# τ(50) — per LMFDB / OEIS https://oeis.org/A000594/list
# Looking at OEIS sequence starting at n=1: ...
# 1, -24, 252, -1472, 4830, -6048, -16744, 84480, -113643, -115920, 534612, -370944,
# -577738, 401856, 1217160, 987136, -6905934, 2727432, 10661420, -7109760, -4219488,
# -12830688, 18643272, 21288960, -25499225, 13865712, -73279080, 24647168, 128406630,
# -29211840, -52843168, -196706304, 134722224, 165742416, -80873520, -182213314, 308120442,
# 101585520, 1133263380, -1041709440, ...
# That's 40 values. Let me trust this list.
# Position 50 is τ(50). Let me get exact value via independent computation using mpmath qfromm.

# Actually, sage has tau. Let me compute via mpmath's nzeros for delta or via the eta product more carefully
# using mpmath with high precision.
mp.mp.dps = 50
# Compute coefficients of η(τ)^24 * q where η = q^{1/24} ∏(1-q^n)
# Just use polynomial computation but with arbitrary integers (which python supports)
# v2 gave 611981400 which differs from claimed 27474750464. Let me check OEIS directly.

# Check: the OEIS listing for A000594:
# Let me just compare τ(8) which I know = 84480. Mine: 84480. Good.
# τ(20) should be -7109760. Mine:
print('τ(20)=', taus200[19])
print('τ(40)=', taus200[39])
# OEIS τ(40) = -1041709440


τ(20)= -7109760
τ(40)= 408038400


In [9]:
# τ(40) is wrong! Mine gives 408038400, OEIS gives -1041709440. 
# So a bug exists in the recursion. But τ(1..20) matched. Let me check the boundary more carefully.
# τ(20) matches, but τ(40) doesn't. Hmm.

# Wait — actually let me check OEIS itself more carefully. Going to https://oeis.org/A000594/b.txt
import urllib.request
try:
 data = urllib.request.urlopen('https://oeis.org/A000594/b.txt', timeout=10).read().decode()
 lines = data.strip().split('\n')[:60]
 for line in lines:
 print(line)
except Exception as e:
 print('Failed:', e)


Failed: HTTP Error 403: Forbidden


In [10]:
# Let me use sage's mpmath alternative or a different package.
# Actually, the cleanest source: τ is multiplicative. τ(p^k+1) = τ(p)τ(p^k) - p^11 τ(p^(k-1))
# τ(2) = -24. τ(4) = τ(2)^2 - 2^11 = 576 - 2048 = -1472. ✓
# τ(8) = τ(2)τ(4) - 2^11 τ(2) = -24*-1472 - 2048*-24 = 35328 + 49152 = 84480. ✓
# τ(16) = τ(2)τ(8) - 2^11 τ(4) = -24*84480 - 2048*-1472 = -2027520 + 3014656 = 987136
print('My τ(16)=', taus200[15])
# τ(32) = τ(2)τ(16) - 2^11 τ(8) = -24*987136 - 2048*84480 = -23691264 - 173015040 = -196706304
print('My τ(32)=', taus200[31])
# τ(64) = τ(2)τ(32) - 2^11 τ(16) = -24*(-196706304) - 2048*987136 = 4720951296 - 2021654528 = 2699296768
print('My τ(64)=', taus200[63])


My τ(16)= 987136
My τ(32)= -196706304
My τ(64)= 2699296768


In [11]:
# OK τ(64) and τ(32) match the recursion. So they're correct! 
# τ(40) = τ(8) τ(5) since gcd(8,5)=1, multiplicative: τ(40) = τ(8) * τ(5) = 84480 * 4830 = 408038400
# That's what my code gives. So my code is RIGHT! Let me check OEIS again.
# Actually OEIS lists: τ(40) might really be 408038400.
# The list I wrote above was wrong. Let me trust the recursion.
print('τ(40) via multiplicativity:', taus200[7] * taus200[4])
# And τ(50) = τ(25) * τ(2)? gcd(25,2)=1.
# τ(25) = τ(5)^2 - 5^11 = 4830^2 - 48828125 = 23328900 - 48828125 = -25499225
# τ(50) = τ(25)*τ(2) = -25499225 * -24 = 611981400. ✓

# Great — my τ values are CORRECT. I was misremembering the OEIS list. Good.
print('τ(50) check:', -25499225 * -24)


τ(40) via multiplicativity: 408038400
τ(50) check: 611981400


In [12]:
# Good, my Ramanujan tau values are correct. Now let me design the corrected Lambda_Delta.
# Strategy:
# 1. Use mpmath quad with [1, mp.inf] OR with [1, y_max] where y_max adapts to t.
# 2. Use higher precision dps=80.
# 3. Test with scipy.integrate.quad as an alternative? Actually mpmath supports infinite bounds.
# 
# Key insight: f_Δ(y) decays like τ(1)*exp(-2πy) = exp(-2πy) for large y.
# y^(s-1) for s = 1/2+it has |y^(s-1)| = y^(-1/2). 
# Similarly y^(11-s) for s = 1/2+it: |y^(11-s)| = y^(10.5). Wait — but this is the analytic 
# normalization where critical line is Re(s)=1/2. Hmm, but the code uses arithmetic normalization
# with s=6+it. Let me re-read the spec.
#
# The spec §2(4): "L(Δ,s) = Σ τ(n) n^{−s}, analytically normalized so critical line is Re(s)=½."
# The completed Λ function: Λ(s) = (2π)^{-s} Γ(s) L(s) with functional eq s → 12-s in arithmetic norm.
# In analytic norm s_an = s_arith - 11/2, so critical line is Re(s_an)=1/2, functional eq s_an → 1-s_an.
#
# The existing code uses arithmetic norm: s = 6 + it where t is the analytic ordinate, since
# 6 = 1/2 + 11/2. Functional equation in arithmetic norm is s → 12-s.
# Λ(s) = ∫_1^∞ (y^{s-1} + y^{11-s}) f_Δ(y) dy is the SYMMETRIC form already satisfying Λ(s)=Λ(12-s).
# For s = 6+it: y^(s-1) = y^(5+it), y^(11-s) = y^(5-it). 
# Magnitude: |y^(5±it)| = y^5. So integrand decays as y^5 * exp(-2πy) for large y.
# 
# That decays VERY fast: at y=12, y^5 * exp(-2πy) = 248832 * exp(-75.4) ≈ 248832 * 1.6e-33 ≈ 4e-28
# At y=20, ≈ 3.2e6 * exp(-125.66) ≈ 6e-49. So y_max=20-25 should be plenty.
#
# But the OSCILLATION: y^(it) has phase t * log(y). For t=100, log(20)=3, so phase ~300 rad,
# leading to massive cancellation. The integrand magnitude is small (y^5 e^{-2πy}), but the 
# oscillations cause numerical cancellation issues if precision is too low.
# 
# So I need: high dps, sufficiently long y_max, robust quadrature.
# 
# Let's test mp.quad with [1, mp.inf]:
mp.mp.dps = 50
taus = taus200[:200]

def f_Delta(y):
 twopi_y = 2*mp.pi*y
 s = mp.mpf(0)
 e_n = mp.exp(-twopi_y)
 e_pow = e_n
 for n in range(1, 201):
 s += taus[n-1] * e_pow
 e_pow *= e_n
 if abs(e_pow) < mp.mpf(10)**(-60):
 break
 return s

def Lambda_Delta_new(t, y_upper=mp.inf):
 s = mp.mpc(6, t)
 integrand = lambda y: (y**(s-1) + y**(11-s)) * f_Delta(y)
 return mp.re(mp.quad(integrand, [1, y_upper]))

# Compare at t=9.222 (known first zero)
import time
t0 = time.time()
v = Lambda_Delta_new(mp.mpf('9.2223793999211025'))
print(f'Λ(9.22238) = {v} ({time.time()-t0:.2f}s)')


Λ(9.22238) = 1.2834435177677708765116324655998395369168452982178e-22 (0.20s)


In [13]:
# Excellent! Λ at first zero is 1.3e-22, essentially zero. Now test at a larger t with mp.inf
import time
t0 = time.time()
v = Lambda_Delta_new(mp.mpf('90.963470616422363')) # 50th zero from existing file
print(f'Λ(90.963) = {v} ({time.time()-t0:.2f}s)')

t0 = time.time()
v = Lambda_Delta_new(mp.mpf('100.0'))
print(f'Λ(100.0) = {v} ({time.time()-t0:.2f}s)')


Λ(90.963) = -3.540240403039822592106312780708447968040967659769e-55 (0.70s)


Λ(100.0) = -4.1272701816933246898290816287166883805057665265741e-55 (0.65s)


In [14]:
# Wait — Λ(100.0) is 4e-55, that's essentially zero! And Λ(90.963) is also 3.5e-55.
# This means mp.quad with [1, inf] is returning ZERO due to insane cancellation.
# That's a problem — we need to scale the integrand or use higher precision.
# 
# Indeed, the integrand magnitude: at y=1, magnitude is f_Delta(1) * y^5 = f_Delta(1) * 1.
# f_Delta(1) = sum τ(n) exp(-2πn). Magnitude ~ exp(-2π) ≈ 1.87e-3. So integrand ~ 1.87e-3 at y=1.
# Then decays. But oscillation y^(it) introduces cancellation. Real part of integral should be ~ same order.
# 
# Λ(s) = (2π)^(-s) Γ(s) L(s) in arithmetic norm. At s=6+it, |Γ(6+it)| ≈ |Γ(6+it)| ~ t^5 e^(-πt/2) sqrt(2π)
# at large t. At t=100: t^5=1e10, e^(-50π)=e^(-157.08)=5e-69. So |Γ(6+it)| ~ 5e-58.
# (2π)^(-6-it) magnitude = (2π)^(-6) ≈ 1.5e-5.
# |L(6+it)| ≈ O(1).
# So |Λ| ~ 1.5e-5 * 5e-58 * O(1) ≈ 1e-62.
# 
# So real Λ at t=100 IS of order 1e-62 or so. So 4e-55 is actually noise above the true value.
# At t=9.22, |Γ(6+9.22i)| is much larger, so Λ is of order 1e-22 there (which matches our zero detection threshold).
# 
# So we need to be careful. We need a NORMALIZED Λ function (Hardy Z-function style) to remove
# the exponential decay. Otherwise root-finding fails at high t.
# 
# Standard trick: use the Hardy Z-function for L(Δ): 
# Z(t) = e^{iθ(t)} L(1/2+it, Δ) where θ chosen so Z is real on critical line.
# For L(Δ), in analytic norm: gamma factor is Γ_C(s+11/2) = 2(2π)^(-s-11/2) Γ(s+11/2)
# So θ(t) = arg(Γ(11/2+1/2+it)/(2π)^(11/2+1/2+it)) /... 
# 
# Better approach: use mpmath's built-in functions! mp.dirichlet or rather the modular form L.
# Actually mpmath doesn't have L(Δ,s) directly. But we can implement Z-function explicitly.
#
# Let me use the analytic normalization L_an(s) = L(s+11/2) and gamma factor Γ_C(s+11/2)=2*(2π)^(-s-11/2)*Γ(s+11/2):
# Λ_an(s) = (2π)^(-s-11/2) Γ(s+11/2) L_an(s), functional eq Λ_an(s) = Λ_an(1-s).
# On critical line s=1/2+it: 
# exponent of (2π): -(6+it) → magnitude (2π)^(-6).
# |Γ(6+it)| ~ exp(-πt/2) * t^(5.5) * sqrt(2π) → huge decay.
# So Λ_an(1/2+it) = (2π)^(-6-it) * Γ(6+it) * L_an(1/2+it)
# 
# Hardy Z(t) := Λ_an(1/2+it) / |gamma factor at 1/2+it|
# = e^{iθ(t)} * L_an(1/2+it) where θ = arg[(2π)^(-6-it) Γ(6+it)]
# 
# Then Z(t) is real and we find sign changes.
# 
# Computing Z(t) requires computing L_an(1/2+it). We can compute L_an(1/2+it) directly via Dirichlet series 
# in analytic norm: L_an(s) = Σ τ(n)/n^(11/2) / n^s.
# Coefficients a_n = τ(n)/n^(11/2). By Ramanujan-Petersson, |a_n| ≤ d(n).
# But series Σ τ(n) n^(-11/2-1/2-it) converges absolutely (since |τ(n)| ≤ n^(11/2+ε)d(n), so a_n n^(-1/2) ≤ d(n)/n^(1/2)
# which is conditionally convergent at best). Need acceleration via approximate functional equation or 
# the SYMMETRIC integral form already in code!
# 
# Key insight: The symmetric integral form Λ(s) = ∫_1^∞ (y^(s-1)+y^(11-s)) f(y) dy IS the 
# approximate functional equation that combines both directions of the Mellin transform.
# But to find zeros we need to remove the exponential decay of Γ(6+it).
# 
# Solution: Divide the symmetric integral by the gamma-factor magnitude:
# Z(t) = Λ(6+it) / |Λ_baseline(6+it)| ... not quite trivial
# 
# Actually, the existing code returns mp.re(integral). At t=9.22, value was 1e-22 which is small 
# enough relative to the integrand magnitude (~1e-3 at y=1) — that's the small "zero" value relative
# to the typical scale. At t=100, the integrand magnitude is much smaller due to the gamma decay,
# and the typical scale is also much smaller (1e-62). So 4e-55 there is large relative to 1e-62.
# 
# What we really need: SIGN of Λ. The mp.re(integral) gives the correct sign as long as numerical
# precision is sufficient. The challenge is when |Λ| ~ 10^(-58) we need dps > 58+ to distinguish.
# At t=100, |Λ| ~ 1e-62, need dps ~ 70-80. 
# At t=200 (for 200th zero ~ 200), |Λ| ~ exp(-πt/2)*... ~ exp(-π*100) ~ 1e-137. 
# Wait, that would be t=200... need dps ~ 145.
# 
# Looking at https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/, the 200th zero
# of L(Δ) is around what value? Spacing density ~ log(t)/(2π) per unit. So 200th zero ~ t such 
# that ∫_0^t (log(τ/2π)+...)/(2π) dτ = 200. Roughly t ~ 220 maybe.

# Better approach: compute Z-function = e^{iθ} L(s) using the Riemann-Siegel-like formula
# OR scale the symmetric integral by exp(πt/2) to remove exponential decay.

# Let me try: rescale the integrand by extracting the dominant exponential phase.
# 
# Actually let's try a much simpler approach: high enough dps to make mp.quad with [1,inf] work.
# At t=200, need dps ~ 150. Let me test.

mp.mp.dps = 100
t0 = time.time()
v = Lambda_Delta_new(mp.mpf('100.0'))
print(f'dps=100, Λ(100) = {v} ({time.time()-t0:.2f}s)')
v = Lambda_Delta_new(mp.mpf('90.963470616422363'))
print(f'dps=100, Λ(90.963) = {v}')


dps=100, Λ(100) = -4.109074336717247896932789213048383617200526113376896528920235551518818024605033878422684255410980985e-55 (1.14s)


dps=100, Λ(90.963) = -3.522122334036778891655950672568623890115646002983947128330927253389134078009610276596685335287233477e-55


In [15]:
# Still 1e-55 magnitude even with dps=100. That suggests the integrand is NOT being computed
# correctly to high precision — likely f_Delta is using mp values but the integrand magnitude is
# inherently larger than true |Λ|, and quad isn't canceling them precisely.
# 
# The issue: y^(s-1)*f(y) at y=1 has magnitude |f(1)| ~ 5e-4. So the integral has summands of
# order 5e-4, and we're trying to get a result of order 1e-62. That requires ~58 digits of 
# CANCELLATION precision in the integrand. mp.quad's adaptive integration should achieve this
# if dps is high enough. Let me try dps=150.

mp.mp.dps = 150
v = Lambda_Delta_new(mp.mpf('100.0'))
print(f'dps=150, Λ(100) = {v}')
v = Lambda_Delta_new(mp.mpf('9.2223793999211025'))
print(f'dps=150, Λ(9.22238) = {v}')


dps=150, Λ(100) = -4.89057854775918888123446849586837469367021789832726871972502656330010989142299135327765412722030838325884767974520548888322542747929940686204133017263e-55


dps=150, Λ(9.22238) = 0.000000000000000000000128344351776777087651163246559984519373355016780651719513120162272253267396619350190600192790432957366401879274099990129341675070425644020011867222448


In [16]:
# The 1e-55 value at t=100 IS the true value! Λ(100+6i) really is ~ 5e-55 nonzero.
# (Wait, no, Λ(6+100i) — i.e., the value at s=6+100i. Not at a zero — 100 is just an arbitrary t.)
# 
# OK so the integral form IS correct! At t=9.22 (a true zero), Λ ≈ 1e-22 — very small relative to
# typical magnitude. At t=100 (not a zero), Λ ≈ 5e-55 — small in absolute but not relative to typical
# scale at that height.
# 
# The TYPICAL scale at t=100: integrand |~ f(1) * y^5 * exp(-2πy)| dy integrated, but with rapid 
# oscillation y^(it), so cancellation produces small result. The non-cancelled magnitude is what
# matters. Expected |Λ(6+it)| ~ |Γ(6+it)| (2π)^(-6) ~ t^(5.5) exp(-πt/2) (2π)^(-6).
# At t=100: 100^5.5 = 1e11, exp(-50π) = 5e-69, (2π)^(-6) = 1.5e-5. Product ~ 7.5e-63. 
# But we get 5e-55. Discrepancy of 1e8 — maybe my asymptotic is wrong or L(Δ) magnitude at height 100
# is large.
# 
# Actually |L(1/2+it,Δ)| can grow polynomially in t — convexity bound t^(1/4). For Hecke L-function 
# with conductor 1: |L(1/2+it)| << t^(1/4+ε). So at t=100, |L| ~ 3-10. 
# Hmm doesn't account for 1e8 factor. Let me just trust the numerical result.
# 
# OK so this means: SIGN CHANGES of Λ(t) determine zeros, and Λ goes from positive to negative or 
# vice versa as t crosses a zero. We just need to detect sign changes reliably.
# 
# At each t, we need dps >= |digits in Λ to identify sign|. If |Λ| ~ 1e-55 at t=100, then dps=80
# gives us only 25 digits of significance for sign — but that should be enough for sign detection
# IF the numerical noise from quad is below 1e-55. Adaptive quad's accuracy is usually 10^(-dps), so
# at dps=80, accuracy is ~1e-80, far below 1e-55. Good.
#
# Let me check: at t=200, expected |Λ| ~ exp(-π*100) ~ 1e-137. Need dps > 137.
# At t=220 (where 200th zero might lie), need dps ~ 150.
# 
# However for SIGN detection, we don't need ALL digits accurate, just enough to determine sign.
# Since quad accuracy is ~10^(-dps), and we need |result| > accuracy, we need dps > |log10(Λ)|.
# 
# Plan: use dps=120 by default; that handles t up to ~170. For larger t, bump up. Let me estimate
# where the 200th zero lies.

# Spacing density at height T: zeros per unit t ≈ log(T/(2π))/(2π) (for L of conductor 1, weight 12,
# γ-factor Γ_C). Approximation: N(T) ≈ T/(2π) (log T - log(2π) + 1) + O(log T).
# So N(T) = 200 → solve. Try T=200: 200/(2π)*(log 200 - log 2π +1) = 31.83 * (5.30 - 1.84 + 1) = 31.83*4.46 = 141.9.
# Try T=300: 300/(2π)*(log 300 - log 2π +1) = 47.75 * (5.70-1.84+1) = 47.75*4.86 = 232.0.
# Try T=260: 260/(2π)*(log 260 - log 2π +1) = 41.38*(5.56-1.84+1) = 41.38*4.72 = 195.3.
# Try T=270: 42.97*(5.598-1.84+1) = 42.97 * 4.76 = 204.5.
# So 200th zero is around T~265-270.
# 
# So need dps to handle t up to ~270. Estimate |Λ(6+270i)| ~ ?
# More carefully: |Λ(6+it)| ~ |Γ(6+it)| * (2π)^(-6) * |L(1/2+it)|
# |Γ(6+it)| via Stirling: |Γ(σ+it)| ≈ sqrt(2π) |t|^(σ-1/2) exp(-π|t|/2) for large t
# At σ=6, t=270: sqrt(2π) ≈ 2.507. |t|^5.5 = 270^5.5 = exp(5.5 ln 270) = exp(5.5 * 5.598) = exp(30.79) = 2.36e13.
# exp(-π*270/2) = exp(-424.12) = 1.5e-184.
# (2π)^(-6) ≈ 1.5e-5
# Product: 2.507 * 2.36e13 * 1.5e-184 * 1.5e-5 ≈ 1.3e-175.
# |L(1/2+it,Δ)| ≤ t^(1/4) * polylog ≈ 5 or so.
# 
# So |Λ| ~ 1e-175 at t=270. Need dps ~ 180-200.

# This is becoming expensive but doable. Let me design a strategy: adaptive dps based on t.
# Alternatively, divide out the gamma factor magnitude (compute Z-function-like form numerically).
# 
# Best approach: Define Z(t) := Λ(6+it) / G(t) where G(t) is the magnitude factor.
# Then Z(t) is order 1 and sign changes are easy.
# 
# G(t) = |Γ(6+it)| * (2π)^(-6) is a positive real number. So Z(t) = Λ(6+it)/G(t) has same sign as
# Λ. Computing Λ and dividing avoids underflow.

# Actually the cleanest approach: compute the integrand but extract exp(πt/2) factor.
# Integrand: (y^(s-1) + y^(11-s)) f_Δ(y). For y = exp(τ) (substitution), 
# y^(it) = exp(it*τ). Hmm but no exponential decay built in.
# 
# Just rescale at the end: compute Λ, then return Λ * exp(πt/2). Since exp(πt/2) is a positive real,
# it preserves sign. Then we can detect sign changes without underflow issues.

# Let me try this approach but with high dps.
def Z_Delta(t, dps_local=None):
 if dps_local is not None:
 mp.mp.dps = dps_local
 s = mp.mpc(6, t)
 integrand = lambda y: (y**(s-1) + y**(11-s)) * f_Delta(y)
 val = mp.re(mp.quad(integrand, [1, mp.inf]))
 return val * mp.exp(mp.pi * mp.mpf(t) / 2)

mp.mp.dps = 50
import time
for t in [9.22, 13.91, 30, 50, 90.96, 100, 150]:
 t0 = time.time()
 z = Z_Delta(mp.mpf(t), dps_local=60)
 print(f't={t}: Z={float(z):+.3e} ({time.time()-t0:.2f}s)')


t=9.22: Z=+2.681e-02 (0.39s)


t=13.91: Z=+4.138e-01 (0.35s)


t=30: Z=+1.514e+04 (0.35s)


t=50: Z=+2.546e+05 (0.35s)


t=90.96: Z=-4.055e+07 (0.77s)


t=100: Z=-6.801e+13 (0.70s)


t=150: Z=+1.941e+48 (0.70s)


In [17]:
# Hmm Z is GROWING with t. That's because I'm extracting too much. Let me look at Stirling more carefully:
# |Γ(6+it)| ~ sqrt(2π) t^(5.5) exp(-π t / 2) for large t (Stirling)
# So Z'(t) := Λ(6+it) / [t^5.5 exp(-π t/2)] should be order 1 (modulo L factor).
# 
# Let me divide by both: Z(t) = Λ * exp(πt/2) / t^(5.5)

def Z_Delta_v2(t, dps_local=None):
 if dps_local is not None:
 mp.mp.dps = dps_local
 s = mp.mpc(6, t)
 integrand = lambda y: (y**(s-1) + y**(11-s)) * f_Delta(y)
 val = mp.re(mp.quad(integrand, [1, mp.inf]))
 # Rescale by inverse magnitude of γ-factor
 t_mp = mp.mpf(t)
 return val * mp.exp(mp.pi * t_mp / 2) / (t_mp**mp.mpf('5.5'))

mp.mp.dps = 50
for t in [9.22, 13.91, 30, 50, 90.96, 100, 150, 200, 250]:
 t0 = time.time()
 z = Z_Delta_v2(mp.mpf(t), dps_local=60)
 print(f't={t}: Z={float(z):+.4e} ({time.time()-t0:.2f}s)')


t=9.22: Z=+1.3253e-07 (0.35s)


t=13.91: Z=+2.1306e-07 (0.36s)


t=30: Z=+1.1379e-04 (0.34s)


t=50: Z=+1.1522e-04 (0.34s)


t=90.96: Z=-6.8280e-04 (0.68s)


t=100: Z=-6.8008e+02 (0.67s)


t=150: Z=+2.0873e+36 (0.69s)


t=200: Z=-2.8039e+69 (0.67s)


t=250: Z=-5.7158e+102 (0.67s)


In [18]:
# Z still growing because at dps=60 we hit numerical noise floor. The true Λ is tiny;
# the value being returned is just numerical noise from mp.quad. 
# 
# At t=100, Z = -6.8e2 vs at t=50, Z = +1.15e-4. That's ratio 6e6. That's way wrong; true Z(t) is bounded.
# This means at t=100, we're already returning NOISE, not the true Λ. dps=60 is insufficient.
# 
# At t=100, true |Λ| ~ 7.5e-63 * |L| ~ 5e-62. Rescaled by exp(πt/2)/t^5.5 = exp(50π)/100^5.5 
# = 1.86e68 / 1e11 = 1.86e57. So true Z ~ 5e-62 * 1.86e57 ~ 9e-5. 
# But we got 6.8e2. So noise floor at dps=60 ~ 1e-60 in Λ, scaled to 6.8e2. Adjust dps.
# 
# At dps=60, mp.quad precision is ~1e-60. Need precision << true |Λ|, e.g. for t=100, need dps > 65.
# For t=200, need dps > 138. For t=270 (200th zero), need dps > 175.
# 
# Let me adapt dps based on t and re-test.

def Z_Delta_v3(t, safety=20):
 t_mp = mp.mpf(t)
 # Required dps: digits for |Λ| ~ exp(-π t /2) factor
 digits_needed = int(float(mp.pi * t_mp / 2 / mp.log(10))) + safety + 10
 mp.mp.dps = max(50, digits_needed)
 s = mp.mpc(6, t_mp)
 integrand = lambda y: (y**(s-1) + y**(11-s)) * f_Delta(y)
 val = mp.re(mp.quad(integrand, [1, mp.inf]))
 return val * mp.exp(mp.pi * t_mp / 2) / (t_mp**mp.mpf('5.5')) if t_mp > 0 else mp.mpf(0)

for t in [9.22, 13.91, 30, 50, 90.96, 100, 150, 200, 250, 270]:
 t0 = time.time()
 z = Z_Delta_v3(mp.mpf(t))
 print(f't={t}: dps={mp.mp.dps} Z={float(z):+.4e} ({time.time()-t0:.2f}s)')


t=9.22: dps=50 Z=+1.3253e-07 (0.16s)
t=13.91: dps=50 Z=+2.1306e-07 (0.16s)


t=30: dps=50 Z=+1.1379e-04 (0.35s)


t=50: dps=64 Z=+1.1522e-04 (0.42s)


t=90.96: dps=92 Z=-6.8280e-04 (1.31s)


t=100: dps=98 Z=-6.8008e+02 (1.04s)


t=150: dps=132 Z=+8.9176e+35 (2.57s)


t=200: dps=166 Z=-3.4280e+69 (6.06s)


t=250: dps=200 Z=+1.2876e+102 (7.11s)


t=270: dps=214 Z=-6.7839e+115 (7.87s)


In [19]:
# Still seeing huge growth. The dps adaptation didn't fix it. Issue: at t=100, dps=98, but Z is still 6.8e2.
# That's bizarre — increasing dps from 60 to 98 didn't change Z(100). Let me think...
# 
# Maybe my asymptotic for Λ is wrong! Let me check what mpmath's L-function thinks.
# Actually mpmath has a generic L-function class. Let me try mp.lerchphi or check whether 
# there's a way to evaluate L(Δ, 1/2+it) directly and verify magnitude.

# Direct partial sum (truncated Dirichlet series) — but it doesn't converge on critical line.
# Let me use the analytic continuation via the integral itself but check carefully.
# 
# Wait — maybe the issue is the integrand at t=100 has REAL oscillations dominating; the integrand
# magnitude at y=1 is f_Delta(1) which is ~ 5e-4, and 5e-4 * exp(50π) = 5e-4 * 1.9e68 = ~1e65.
# At dps=98, we have 98 digits, so subtraction precision allows results down to ~ 1e65 * 1e-98 = 1e-33.
# Original Λ is real of order ... let me just measure the integrand at a few points.

# Let me check: is Λ actually returning a value of order 1e-55 or noise?
mp.mp.dps = 100
s = mp.mpc(6, 100)
def integrand_at(y):
 return (y**(s-1) + y**(11-s)) * f_Delta(y)
print('integrand(1) =', integrand_at(mp.mpf(1)))
print('integrand(2) =', integrand_at(mp.mpf(2)))
print('integrand(5) =', integrand_at(mp.mpf(5)))
print('integrand(10) =', integrand_at(mp.mpf(10)))


integrand(1) = (0.003570739701284303808686109920684524621162219727232761148681702282696029522079612001432606723726066441 + 0.0j)
integrand(2) = (0.0002187368559904455259898776359489295507450166667073688346532402521038972660217941367834478870717629846 + 0.0j)
integrand(5) = (-0.0000000001064736657720893699331049279457579703232015455813183362008296223763617202810430611170865398714860237 + 0.0j)
integrand(10) = (-0.00000000000000000000006231073770463781456937241804067871117358009054910524000817891276926515232272145244413452436346656425 + 0.0j)


In [20]:
# All integrand values are purely real (imaginary = 0)? That's because s=mpc(6,100) and y is real positive,
# so y^(s-1) = exp((s-1) ln y) = exp(5 ln y + 100i ln y). It SHOULD be complex.
# Let me check more carefully.
print('s =', s)
print('y^(s-1) at y=2:', mp.mpf(2)**(s-1))
print('y^(11-s) at y=2:', mp.mpf(2)**(11-s))
# the sum (y^(s-1)+y^(11-s)) should be 2*Re(y^(s-1)*y^5 something). Actually:
# y^(s-1) + y^(11-s) = y^(5+it) + y^(5-it) = y^5 * (y^(it) + y^(-it)) = 2 y^5 cos(t ln y).
# So integrand = 2 y^5 cos(t ln y) f_Delta(y). Real always!


s = (6.0 + 100.0j)
y^(s-1) at y=2: (31.36416530917305239135343192785521987612736472219300399449606409648880985867374354235062775461784392 + 6.347372248329669761933881166823758083119758328712645553637603874297405662422719255656807926765991956j)
y^(11-s) at y=2: (31.36416530917305239135343192785521987612736472219300399449606409648880985867374354235062775461784392 - 6.347372248329669761933881166823758083119758328712645553637603874297405662422719255656807926765991956j)


In [21]:
# Yes the sum is real ≈ 2 * 31.36 = 62.7 at y=2. So integrand is real and oscillates.
# Magnitude at y=1: 0.0036, at y=2: 0.00022, at y=5: 1e-10, at y=10: 6e-23. All small. 
# So Λ(6+100i) should be of order 0.001 or so. Let me recompute:

mp.mp.dps = 100
val = mp.quad(lambda y: integrand_at(y), [1, mp.inf])
print('Λ(6+100i) =', val)
print('|Λ| =', abs(val))


Λ(6+100i) = (-4.109074336717247896932789213048383617200526113376896528920235551518818024605033878422684255410980985e-55 + 0.0j)
|Λ| = 4.109074336717247896932789213048383617200526113376896528920235551518818024605033878422684255410980985e-55


In [22]:
# So integrand at y=1 is ~0.0036 but the integral over [1,inf] is 4e-55. 
# That's MASSIVE cancellation from the oscillation 2 y^5 cos(t ln y) f(y) — ~58 orders of cancellation!
# That's because t=100 makes the cosine oscillate ~ 100/(2π) ≈ 16 times per unit decade of y.
# The integral really should be tiny.
# 
# But wait — my earlier asymptotic estimate said |Λ| at t=100 ~ 5e-62, not 5e-55. Let me re-check Stirling.
# |Γ(σ+it)| for large t: Γ(σ+it) ≈ sqrt(2π) (σ+it)^(σ+it-1/2) e^(-(σ+it))
# = sqrt(2π) |σ+it|^(σ-1/2) exp((σ-1/2) i arg + i arg*it - σ - it) ...
# More carefully: |Γ(σ+it)| ≈ sqrt(2π) |t|^(σ-1/2) exp(-π|t|/2) for large t (the standard asymptotic).
# At σ=6, t=100: sqrt(2π) * 100^(5.5) * exp(-50π) = 2.507 * 10^11 * 1.86e-69 = 4.67e-58
# (2π)^(-6) = 1.5e-5. 
# Product: 7e-63. 
# So |Λ(6+100i)| ~ 7e-63 * |L(1/2+100i)|. We get 4e-55. So |L(1/2+100i)| ~ 6e7. That's way too large.
# 
# But L(1/2+it,Δ) is bounded by t^(1/4) for cusp forms (Lindelöf). Realistically |L| ~ 1-10. So |Λ|
# should be 7e-62, not 4e-55. 
# 
# That suggests the integral IS contaminated by numerical noise at dps=100. The true value of |Λ|
# could be even smaller than 4e-55, and we're seeing dps-truncation residue.
# 
# Let me test with dps=200 and see if Λ at t=100 changes.

mp.mp.dps = 200
s = mp.mpc(6, 100)
def integrand_at(y):
 return (y**(s-1) + y**(11-s)) * f_Delta(y)
val = mp.quad(lambda y: integrand_at(y), [1, mp.inf])
print('dps=200 Λ(6+100i) =', val)


dps=200 Λ(6+100i) = (-0.00000000000000000000000000000000000000000000000000000048905785477591888812344684958683746936702178983272687197250265633001098914229913532776541272203083832575985124153527674307412511476881832703963311118804802659627918700457302841015296242020343704937567 + 0.0j)


In [23]:
# At dps=200, Λ(6+100i) ≈ -4.89e-55. Different from the dps=100 value of -4.11e-55! Hmm.
# Wait actually they differ. Let me try dps=150.
mp.mp.dps = 150
s = mp.mpc(6, 100)
def integrand_at(y):
 return (y**(s-1) + y**(11-s)) * f_Delta(y)
val = mp.quad(lambda y: integrand_at(y), [1, mp.inf])
print('dps=150 Λ(6+100i) =', val)

# And dps=300
mp.mp.dps = 300
val = mp.quad(lambda y: integrand_at(y), [1, mp.inf])
print('dps=300 Λ(6+100i) =', val)


dps=150 Λ(6+100i) = (-4.89057854775918888123446849586837469367021789832726871972502656330010989142299135327765412722030838325884767974520548888322542747929940686204133017263e-55 + 0.0j)


dps=300 Λ(6+100i) = (-0.000000000000000000000000000000000000000000000000000000461573163091681275277322766229015582729472671071612979459811834203563960553057997587206480584650039541285216050649847061852955667666603389799997060979203299947060076169822779399451462969744192308188434463513289486099739044228918274903993843491507149751700930574221231760308266504664519819797207739059 + 0.0j)


In [24]:
# At dps=300, Λ(6+100i) = -4.616e-55. Different again! So mp.quad isn't converged.
# This is because we have an oscillatory integral with massive cancellation and mp.quad has 
# limited adaptive capability. dps=150 says -4.89e-55, dps=300 says -4.62e-55. So uncertainty ~3e-56.
# That's the true |Λ|, around 4-5e-55. So |L(1/2+100i)| ≈ 4e-55/7e-63 ~ 6e7? Still wrong.
#
# OR my Stirling asymptotic is missing factors. Let me numerically check |Γ(6+100i)|:
mp.mp.dps = 50
g = mp.gamma(mp.mpc(6, 100))
print('|Γ(6+100i)| =', abs(g))
print('vs Stirling sqrt(2π)*100^5.5*exp(-50π) =', float(mp.sqrt(2*mp.pi))*1e11*float(mp.exp(-50*mp.pi)))


|Γ(6+100i)| = 1.5186772613880364236714027914617751613932112955745e-57
vs Stirling sqrt(2π)*100^5.5*exp(-50π) = 1.5145103377471874e-57


In [25]:
# OK |Γ(6+100i)| = 1.52e-57, Stirling agrees.
# So |Λ(6+100i)| = (2π)^(-6) * |Γ(6+100i)| * |L(1/2+100i)| = 1.5e-5 * 1.52e-57 * |L|
# = 2.3e-62 * |L|.
# Observed |Λ| ≈ 4.6e-55. So |L| ≈ 2e7. 
# 
# That's huge. Maybe L(Δ) has a large value at t=100? Let me look up: LMFDB shows L(1/2+it) for L(Δ)
# Actually, in the analytic normalization L_an(s), critical line is s=1/2, and the relation is
# L_an(s) = L_arith(s + 11/2). So L_arith(1/2 + 11/2 + it) = L_arith(6 + it) = L_an(1/2+it).
# The Dirichlet series Σ τ(n) n^{-(6+it)} converges absolutely (since |τ(n)| ≤ d(n) n^(11/2), 
# Σ |τ(n)|n^(-6) ≈ Σ d(n) n^(-1/2) which doesn't converge). Hmm actually Σ τ(n) n^(-s) converges
# for Re(s) > 13/2, so at s=6+it series doesn't converge — that's the whole reason we need the
# integral representation.
# 
# OK regardless: the integral form gives the analytic continuation. The numerical issue is real:
# adaptive quadrature can't reach 60+ digits of cancellation precision easily. 
# 
# Let me try a different strategy: change variables y = e^u so the integration becomes [0, inf)
# with integrand exp(-2π e^u) e^(u*6) cos(t*u) * τ-sum + correction.
# Or use scipy's QUADPACK with oscillatory weight (qawo).

# But actually, the MAIN POINT is: we just need to find sign changes. The relative magnitude
# isn't critical. The issue is whether sign is reliable.
# 
# At t=100, three runs gave Λ = -4.11e-55 (dps=100), -4.89e-55 (dps=150), -4.62e-55 (dps=300).
# All NEGATIVE. The sign is consistent! So sign change detection works even though we don't have
# precise magnitude.
# 
# But around a true zero, |Λ| is even smaller (literally 0). So sign detection might miss true zeros
# if our numerical noise is ~5e-55 in absolute value. The true zeros have nearby |Λ| values that 
# decrease through 0, and noise of ~5e-55 wouldn't necessarily prevent sign detection — Λ should
# cross from +X to -X with X >> noise in a small region around the zero unless we're literally 
# AT the zero.
# 
# OK let me just try generating zeros and see how it goes. Use the symmetric integral form 
# (already in the code as Lambda_Delta) but with y_upper=mp.inf instead of y_max=12.
# Use dps=80 initially, bump higher for higher t.

# Actually, let me think differently. The existing 50 reliable zeros were generated with dps=80, y_max=25.
# What if I use the exact same approach but extend? Or what if I use the symmetric integral with 
# mp.inf and check that 50 zeros are reproduced?

mp.mp.dps = 80
def Lambda_Delta_inf(t, taus_arr, N=200):
 """Λ(Δ, 6+it) via symmetric Mellin, integration to infinity."""
 s = mp.mpc(6, t)
 def integrand(y):
 return (y**(s-1) + y**(11-s)) * f_Delta_func(y, taus_arr, N)
 return mp.re(mp.quad(integrand, [1, mp.inf]))

def f_Delta_func(y, taus_arr, N):
 twopi_y = 2*mp.pi*y
 s = mp.mpf(0)
 e_n = mp.exp(-twopi_y)
 e_pow = e_n
 for n in range(1, N+1):
 s += taus_arr[n-1] * e_pow
 e_pow *= e_n
 if abs(e_pow) < mp.mpf(10)**(-mp.mp.dps - 20):
 break
 return s

# Verify first zero
import time
mp.mp.dps = 80
t0 = time.time()
v = Lambda_Delta_inf(mp.mpf('9.2223793999211025'), taus200, 200)
print(f'Λ at first zero = {v} ({time.time()-t0:.2f}s)')

# Check value at slightly off-zero point
v1 = Lambda_Delta_inf(mp.mpf('9.1'), taus200, 200)
v2 = Lambda_Delta_inf(mp.mpf('9.3'), taus200, 200)
print(f'Λ(9.1) = {v1}')
print(f'Λ(9.3) = {v2}')


Λ at first zero = 0.00000000000000000000012834435177677708765116324655998587192922234532970423645976787106488217653801443 (0.60s)


Λ(9.1) = 0.00000078812638419724137930392604938521247018151727967014986523152647140297426357681428
Λ(9.3) = -0.00000041721502788236653588603606580930260553884454411638307343759127292772386094040672


In [26]:
# Good. Λ(9.1) > 0, Λ(9.3) < 0, sign change between 9.1 and 9.3 — first zero at 9.222 lies in there. ✓
# 
# Now plan: 
# 1. Write a corrected _zeros_v2.py with new Lambda_Delta (integration [1, inf]) and 
# adaptive dps based on t.
# 2. Generate first 200 zeros.
# 3. Validate against LMFDB or against existing 50 reliable zeros.
# 
# But generating 200 zeros at dps up to 200 for t~270 will be EXPENSIVE.
# 
# Estimate: per Λ evaluation at dps=80, ~0.3s. At dps=200, ~5s. For 200 zeros with ~10 evaluations
# each (scan step 0.2, plus bisection), need ~2000 evaluations. Average maybe 2s each? 4000s.
# Too slow.
# 
# Strategy: use the Hardy Z-function with proper magnitude normalization. The cleanest way is to
# write Λ(6+it) = G(t) * Z(t) where G(t) > 0 is a positive real factor and Z(t) is bounded.
# Then Z(t) and Λ(6+it) have the same sign, and we can detect sign changes of Z(t).
# 
# Define Z(t) = Λ(6+it) / G(t), where G(t) is e.g. |Γ(6+it) * (2π)^(-6-it)| = |Γ(6+it)| * (2π)^(-6)
# That's a positive real.
# 
# How to compute Z(t) directly via integral without exponential underflow?
# Λ(6+it) = ∫_1^∞ (y^(s-1)+y^(11-s)) f_Δ(y) dy = ∫_1^∞ 2 y^5 cos(t ln y) f_Δ(y) dy
# 
# Z(t) = Λ(6+it) / G(t). We need to factor out G(t) at the integrand level to avoid underflow.
# But the integrand doesn't have G(t) explicitly; G(t) emerges from cancellation in the integral.
# 
# So really we need to either:
# (a) compute Λ at very high precision (expensive)
# (b) compute Z(t) via a different formula
# 
# For (b): The standard approach for Hardy Z-function uses the Riemann-Siegel formula. For 
# modular L-functions, there's an analog: approximate functional equation
# Z(t) = (2 Re Σ_{n≤A} a(n)/n^(1/2+it) * exp(-iθ(t))) + ... main terms
# where A ~ sqrt(N q / (2π)^2 ... ) — the conductor-dependent cutoff.
# 
# For L(Δ): conductor is 1 (level 1). Approximate functional equation:
# Λ(s) = Σ_n a_n F_+(s, n) + ε Σ_n a_n F_-(1-s, n) where F are incomplete gamma functions.
# 
# This is the standard, much faster approach. mpmath actually has this for general L-functions!
# Let me check if there's a way to use mpmath for L(Δ) directly.

# Actually mpmath has 'lerchphi' for some things but doesn't have built-in modular L-function.
# Sympy/Sage have it. Let me check whether sympy can compute L(Δ, s).

# Looking at mpmath: there's mp.zeta, mp.dirichlet (Dirichlet L), but no modular L.

# Let me just use a smarter integration. Use scipy.integrate.quad with the qawo weight for 
# oscillatory integrals, or use the substitution y = e^u to turn it into ∫ 2 e^(6u) cos(t u) f_Δ(e^u) du 
# from 0 to infinity. Then use scipy quad with qawo for cosine oscillation.
# 
# But scipy.integrate.quad uses double precision — not enough.
# 
# OK pragmatic approach: use the existing approach but with mp.inf and dps tuned to t.
# Speed estimate: with dps=80 most evaluations should be fast. Let me estimate per-zero scan time.
# Step 0.2, so for t in [0,270] we have ~1350 evaluations, plus ~10 bisection steps per zero (1500 more).
# At dps=80, ~0.5s/eval, so ~25 minutes. At dps=200 (for higher t), ~5s/eval. So maybe 1-2 hrs total.
# 
# Better plan: load the existing 50 zeros file as a starting point. Generate from zero 51 onward.
# But the 50 zeros file uses y_max=25 which the description says is "still insufficient for t > 90".
# Actually wait — the description says t>90 needs higher precision. The 50th zero is at t=90.96.
# So we can trust the 50 zeros file up to t=90.
# 
# Let me first run my corrected approach on the first 10 zeros and validate against the file + LMFDB.

# Speed up: precompute taus and f_Delta values? f_Delta evaluation is expensive (200 exponentials per call).
# Each integrand call: 2 exponentiations of y^(s-1) etc + f_Delta call. Bottleneck is f_Delta.
# Can we cache f_Delta(y) values per quadrature node? mp.quad uses adaptive Gauss-Legendre — same 
# nodes get reused across multiple integrations if y_upper is fixed. Let me cache!

from functools import lru_cache

_taus_global = taus200
_N_global = 200

@lru_cache(maxsize=10000)
def f_Delta_cached(y_str):
 y = mp.mpf(y_str)
 twopi_y = 2*mp.pi*y
 s = mp.mpf(0)
 e_n = mp.exp(-twopi_y)
 e_pow = e_n
 for n in range(1, _N_global+1):
 s += _taus_global[n-1] * e_pow
 e_pow *= e_n
 if abs(e_pow) < mp.mpf(10)**(-mp.mp.dps - 20):
 break
 return s

# But y_str depends on dps and y value — caching is fragile.
# Use a simple dict cache keyed by (str(y), dps).
_f_cache = {}
def f_Delta_smart(y):
 key = (str(y), mp.mp.dps)
 if key in _f_cache:
 return _f_cache[key]
 twopi_y = 2*mp.pi*y
 s = mp.mpf(0)
 e_n = mp.exp(-twopi_y)
 e_pow = e_n
 for n in range(1, _N_global+1):
 s += _taus_global[n-1] * e_pow
 e_pow *= e_n
 if abs(e_pow) < mp.mpf(10)**(-mp.mp.dps - 20):
 break
 _f_cache[key] = s
 return s

# Actually mp.quad's nodes are generated on the fly each call so caching by str(y) might help
# if nodes coincide across calls. Let's just test speed first without caching.
print("Speed test:")
mp.mp.dps = 80
t0 = time.time()
for t in [9.22, 13.91, 17.44, 19.66, 22.34]:
 v = Lambda_Delta_inf(mp.mpf(t), taus200, 200)
print(f'5 evals @ dps=80: {time.time()-t0:.2f}s')


Speed test:


5 evals @ dps=80: 2.73s


In [27]:
# 0.55s per eval at dps=80. Acceptable.
# Let me time at higher dps and higher t.
mp.mp.dps = 120
t0 = time.time()
for t in [150, 160, 170, 180, 190]:
 v = Lambda_Delta_inf(mp.mpf(t), taus200, 200)
print(f'5 evals @ dps=120, t~150-190: {time.time()-t0:.2f}s')

mp.mp.dps = 180
t0 = time.time()
for t in [250, 260, 270]:
 v = Lambda_Delta_inf(mp.mpf(t), taus200, 200)
print(f'3 evals @ dps=180, t~250-270: {time.time()-t0:.2f}s')


5 evals @ dps=120, t~150-190: 17.51s


3 evals @ dps=180, t~250-270: 33.45s


In [28]:
# 3.5s/eval at dps=120, 11s/eval at dps=180. With ~1400 evaluations expected, this is 5-15+ hours.
# Too slow.
# 
# I need a much faster method. Options:
# A) Approximate functional equation with incomplete Γ-function — much smaller cutoff, fewer terms.
# B) Use a Hardy Z-style direct sum.
# C) Switch from mp.quad to a smoothed/oscillation-aware quadrature like Filon's method.
# 
# Actually, the symmetric integral formula uses the TRUE functional equation symmetry. The reason
# it converges quickly is that f_Δ(y) decays exponentially. The issue isn't convergence — it's that
# OUR FACTOR exp(πt/2) is hidden in the cancellation.
# 
# Best approach: change of variable y = e^u. Then dy = e^u du, and
# (y^(s-1) + y^(11-s)) f_Δ(y) dy = (e^(5u+itu) + e^(5u-itu)) f_Δ(e^u) e^u du
# = 2 e^(6u) cos(tu) f_Δ(e^u) du
# Integral becomes ∫_0^∞ 2 e^(6u) cos(tu) f_Δ(e^u) du.
# 
# Now substitute v = e^u, no wait that's what we had. The point is to recognize this as the cosine
# transform of g(u) := 2 e^(6u) f_Δ(e^u). g(u) for large u: 2 e^(6u) τ(1) e^(-2π e^u) = 2 e^(6u-2π e^u).
# Decays super-exponentially as u increases. For u=0, g(0) = 2 f_Δ(1) ≈ 0.00714.
# 
# This is a smooth, super-rapidly decaying function. Its cosine transform Λ(t) ~ exp(-πt/2) by 
# standard Fourier asymptotic of analytic functions.
# 
# To get the Z-function (no decay), divide by exp(-πt/2):
# Z(t) := exp(πt/2) * ∫_0^∞ g(u) cos(tu) du
# But the cosine transform of g(u) has a STATIONARY PHASE point where d/du[tu - πt e^u/something]=0...
# 
# This is getting complex. Let me try a completely different angle: USE mpmath's pre-built tools.
# Specifically, mpmath has `mp.lerchphi`, doesn't help.
# 
# IDEA: Use mpmath's findroot directly on Λ_Delta with brackets. The findroot is fast given good 
# brackets. We only need to evaluate Λ enough times to scan and refine.

# But we still need many Λ evaluations.

# LATERAL IDEA: Use the existing 50 zeros file (already validated) and only compute zeros 51-200.
# That cuts time roughly in half.
# 
# Even better: use the approximate functional equation (AFE). For L(Δ) with conductor 1:
# Λ(s) = Σ_{n≥1} a_n n^(-s) Γ(s, 2πn) /(2π)^(-s) * something + Σ_{n≥1} a_n n^(s-12) Γ(12-s, 2πn) * something
# where a_n = τ(n) and Γ(s,x) is the upper incomplete gamma function.
# 
# Specifically: Λ(s) = ∫_1^∞ (y^(s-1) + y^(11-s)) f(y) dy
# = Σ_n τ(n) [∫_1^∞ y^(s-1) e^(-2πny) dy + ∫_1^∞ y^(11-s) e^(-2πny) dy]
# = Σ_n τ(n) [(2πn)^(-s) Γ(s, 2πn) + (2πn)^(s-12) Γ(12-s, 2πn)]
# 
# Each upper incomplete Γ(s, 2πn) for large n decays as exp(-2πn). So series converges very fast.
# Only need ~30 terms total!
# 
# AND: incomplete gamma function Γ(s, x) for fixed x and growing |s| has known asymptotic. mpmath
# has mp.gammainc(s, a, b, regularized=False).

print('mpmath has gammainc:', hasattr(mp, 'gammainc'))
mp.mp.dps = 50
print(mp.gammainc(mp.mpc(6, 100), 2*mp.pi))


mpmath has gammainc: True
(-1.1497341268944692497970560429330958410876344373302 - 0.0019629537471673494566943910473148288705652673196273j)


In [29]:
# Excellent! mp.gammainc(s, a) computes Γ(s,a) = ∫_a^∞ t^(s-1) e^(-t) dt.
# Let's use the AFE form. 
# 
# Λ(s) = Σ_n τ(n) [(2πn)^(-s) Γ(s, 2πn) + (2πn)^(s-12) Γ(12-s, 2πn)]
# 
# Each term: τ(n) is integer (manageable size up to 200), (2πn)^(-s) is complex magnitude (2πn)^(-Re s) 
# = (2πn)^(-6) for Re s=6. Γ(s, 2πn) ~ (2πn)^(s-1) e^(-2πn) for large 2πn relative to |s|.
# So each term has magnitude τ(n) * (2πn)^(-6) * (2πn)^(5) e^(-2πn) = τ(n) (2πn)^(-1) e^(-2πn).
# For n=10, τ(10)=-115920, (2π*10)^(-1) = 0.0159, e^(-2π*10) = e^(-62.8) = 5e-28. 
# Product: 1.15e5 * 0.0159 * 5e-28 ≈ 9e-25. Tiny.
# 
# So the series converges in ~10-20 terms for any t. SUPER fast.
# 
# Critical observation: there's NO oscillatory cancellation in the n-sum — each term is just complex
# valued. We just sum them up. The exponential decay e^(-πt/2) appears naturally inside Γ(s, 2πn)
# for n=1 since Γ(6+it, 2π) has the dominant phase exp(-πt/2) baked in.

def Lambda_Delta_afe(t, taus_arr, N_terms=40):
 """Λ(Δ, 6+it) via approximate functional equation (Hecke integral series form).
 Λ(s) = Σ_n τ(n) [(2πn)^(-s) Γ(s, 2πn) + (2πn)^(s-12) Γ(12-s, 2πn)]
 Returns real-valued Λ(6+it).
 """
 s = mp.mpc(6, t)
 twopi = 2*mp.pi
 total = mp.mpc(0)
 for n in range(1, N_terms+1):
 a = twopi * n
 term = taus_arr[n-1] * (a**(-s) * mp.gammainc(s, a) + a**(s-12) * mp.gammainc(12-s, a))
 total += term
 return mp.re(total)

mp.mp.dps = 50
t0 = time.time()
v = Lambda_Delta_afe(mp.mpf('9.2223793999211025'), taus200, 40)
print(f'AFE Λ(9.22238)= {v} ({time.time()-t0:.3f}s)')

t0 = time.time()
v = Lambda_Delta_afe(mp.mpf('100'), taus200, 40)
print(f'AFE Λ(100)= {v} ({time.time()-t0:.3f}s)')

t0 = time.time()
v = Lambda_Delta_afe(mp.mpf('250'), taus200, 40)
print(f'AFE Λ(250)= {v} ({time.time()-t0:.3f}s)')


AFE Λ(9.22238)= 1.2834435177677708765116324655998630480821086701556e-22 (0.268s)


AFE Λ(100)= -1.0791173479105980361086478500321721240230347173039e-56 (0.376s)
AFE Λ(250)= -8.8328806342245868865174970919406003165435365735522e-57 (0.188s)


In [30]:
# BLAZING fast! 0.2-0.4s/eval at dps=50. And Λ(9.22)=1.28e-22 matches the integral form exactly!
# 
# But Λ(100) gives 1.08e-56 vs integral form's ~4.6e-55. Hmm discrepancy of ~50x. Need more dps or terms.
# 
# Actually wait: Λ(100) at t=100 might just have huge variation due to oscillation. Let me check 
# at small t with more confidence:
mp.mp.dps = 80
print('AFE Λ(9.22)=', Lambda_Delta_afe(mp.mpf('9.2223793999211025'), taus200, 40))
print('AFE Λ(13.91)=', Lambda_Delta_afe(mp.mpf('13.907549861392134'), taus200, 40))
print('AFE Λ(17.44)=', Lambda_Delta_afe(mp.mpf('17.442776978234473'), taus200, 40))

# These should all be ~ 0 (zero locations). Compare to integral form:
print()
print('INT Λ(9.22)=', Lambda_Delta_inf(mp.mpf('9.2223793999211025'), taus200, 200))


AFE Λ(9.22)= 0.00000000000000000000012834435177677708765116324655998587192922234532970423645976787090889033549766938


AFE Λ(13.91)= -0.000000000000000000000022315257190454196211175356632837435746450295391005044349637278121409527441701475


AFE Λ(17.44)= 0.00000000000000000000000022259587561634417572848979375938148344264342685261407219307385575126981954585845



INT Λ(9.22)= 0.00000000000000000000012834435177677708765116324655998587192922234532970423645976787106488217653801443


In [31]:
# PERFECT. AFE and integral form match exactly (to dps). Both give ~ 1.28e-22 at first zero — 
# meaning the actual zero is very close to 9.2223793999211025 but our value of t differs from the 
# true zero by enough to give a residual ~ 1e-22.
# 
# The Λ values at "zero" locations are nonzero because the ordinates have ~15 decimal precision only.
# 
# Speed at dps=80: ~0.3-0.5s/eval. Need to check at higher t with more terms required:

mp.mp.dps = 80
t0 = time.time()
v = Lambda_Delta_afe(mp.mpf(270), taus200, 40)
print(f'Λ(270) @ dps=80 = {v} ({time.time()-t0:.3f}s)')

# How many terms do we need? Let's check convergence:
mp.mp.dps = 80
t_test = mp.mpf(270)
vals = []
for N in [10, 20, 30, 40, 50, 80, 100]:
 if N > 200: continue
 v = Lambda_Delta_afe(t_test, taus200, N)
 vals.append((N, v))
 print(f'N={N}: Λ(270)={v}')


Λ(270) @ dps=80 = -1.9841312315426617628643707906367398106447411089594148251808345158561347450848319e-87 (0.318s)
N=10: Λ(270)=-8.463761376182067248232332459111532509619707350587465104940989698005469544824281e-28
N=20: Λ(270)=6.0040979358810005731007507126598639849701122629339062610237228503567066557999272e-54


N=30: Λ(270)=4.7578602633373277829736209889038916482335126751201373754367562465689867105477898e-80


N=40: Λ(270)=-1.9841312315426617628643707906367398106447411089594148251808345158561347450848319e-87


N=50: Λ(270)=-1.984131231542661762713495461743907950807302597889154508629636013624997850842869e-87


N=80: Λ(270)=-1.984131231542661762713495461743907950807302598323681114184825649957274081825935e-87


N=100: Λ(270)=-1.984131231542661762713495461743907950807302598323681114184825649957274081825935e-87


In [32]:
# So at t=270 with dps=80, N=50 terms is converged. At t=270 |Λ| ~ 2e-87, less than 10^(-80), 
# so we need MORE dps. Let me bump dps.

mp.mp.dps = 120
t_test = mp.mpf(270)
import time
for N in [40, 50, 60, 80]:
 t0 = time.time()
 v = Lambda_Delta_afe(t_test, taus200, N)
 print(f'dps=120 N={N}: Λ(270)={v} ({time.time()-t0:.2f}s)')


dps=120 N=40: Λ(270)=-1.50875328892831859837677553503573039227987777684305249092237183919175991014978473384189872054590103629330789370445711545e-106 (0.61s)


dps=120 N=50: Λ(270)=-2.39042433312722676789275453168354850274274922439404339481904197315971451849921552179753918387555125446952845069588850494e-127 (0.78s)


dps=120 N=60: Λ(270)=-2.39042867839328231978911785444586691691595853209026376770301770707620183285173378774551939563367936620197125486095352215e-127 (0.98s)


dps=120 N=80: Λ(270)=-2.39042867839328231978911785444585833340307784538987787828879414539525987551555184234308019786028339415188345659278229549e-127 (2.02s)


In [33]:
# At dps=120, N=60 converges Λ(270) to ~ -2.39e-127. 1 second per eval. 
# At t=270 the typical scale is exp(-π*270/2) ~ exp(-424) ~ 1e-184. So |Λ| ~ 1e-127 means typical 
# baseline factor is much smaller — but that's fine since L_an(1/2+it,Δ) magnitude varies a lot.
# 
# Plan finalize: adaptive dps based on t. Estimate floor: |Λ(6+it)| ~ exp(-πt/2) * (...).
# Required dps > log10(exp(πt/2)) = πt/(2 ln 10) ≈ 0.682 t. Plus safety margin 30.
# 
# For t up to 270: dps = 0.682*270 + 30 = 214. Let me use dps = ceil(0.7*t + 30) and N_terms 
# adaptive too.
# 
# Speed at dps=214, N=80: roughly... let me test.

mp.mp.dps = 220
t_test = mp.mpf(270)
import time
t0 = time.time()
v = Lambda_Delta_afe(t_test, taus200, 60)
print(f'dps=220 N=60: Λ(270)={v} ({time.time()-t0:.2f}s)')

t0 = time.time()
v = Lambda_Delta_afe(t_test, taus200, 80)
print(f'dps=220 N=80: Λ(270)={v} ({time.time()-t0:.2f}s)')


dps=220 N=60: Λ(270)=-8.583512880686699220238514863224203652708303982255502039883485654633973939913420632181235065707129342700376783986404961081215972246957304411546756886933387800228738229186237542991568113737088157283824358274521201656100369e-160 (3.17s)


dps=220 N=80: Λ(270)=1.165650899360337477289249032199689900399314287741338076147884847539045424681630656967154765649274299150086179625873431829099646054547361031584325696929923241720325587202459790596297451772911632532197950883470464232581982e-175 (6.01s)


In [34]:
# Hmm at dps=220 N=80, Λ has changed sign and become 1e-175 (not -8e-160). Need more terms or 
# something's not converging here. Let me check N=100, 120, 150.

mp.mp.dps = 220
t_test = mp.mpf(270)
import time
for N in [100, 120, 150, 180]:
 t0 = time.time()
 v = Lambda_Delta_afe(t_test, taus200, N)
 print(f'dps=220 N={N}: Λ(270)={v} ({time.time()-t0:.2f}s)')


dps=220 N=100: Λ(270)=1.165650899360337477289249032199689900399791084847111923979106330533019268166303103527267697212228426373855362600658687377455482590546436526708310335542932968724552763952039080886611779455016889796491931514871630946111576e-175 (10.45s)


dps=220 N=120: Λ(270)=1.165650899360337477289249032199689900399791084847111923979106330533019268166303103527267697217695229801008575381266128891688449093224059177313916383331310864721599841613173126832538546733761732838384026359451320361608757e-175 (15.97s)


dps=220 N=150: Λ(270)=1.16565089936033747728924903219968990039979108484711192397910633053301926816630310352726769721769522980100857538126612889168844909322405917731391638333424716140191668701671549046868961173040968544325166898001594551193794e-175 (26.58s)


dps=220 N=180: Λ(270)=1.16565089936033747728924903219968990039979108484711192397910633053301926816630310352726769721769522980100857538126612889168844909322405917731391638333424716140191668701671549046868961173040968544325166898001594551193794e-175 (26.92s)


In [35]:
# So at dps=220, N=120 converges to Λ(270)=1.17e-175. Cost ~ 16s/eval. For 200 zeros at ~10 evals each,
# 2000 * 16s = 32000s = 9 hours. Still too slow.
# 
# Key optimization: dps doesn't need to be that high. The mp.gammainc(s, a) for |s| large and 
# a small has dominant magnitude exp(-πt/2)... most of which cancels out. Let me check if 
# gammainc is the bottleneck.

import time
mp.mp.dps = 220
s = mp.mpc(6, 270)
t0 = time.time()
for n in range(1, 30):
 v = mp.gammainc(s, 2*mp.pi*n)
print(f'30 gammainc evals @ dps=220: {time.time()-t0:.2f}s')

mp.mp.dps = 80
t0 = time.time()
s = mp.mpc(6, 270)
for n in range(1, 30):
 v = mp.gammainc(s, 2*mp.pi*n)
print(f'30 gammainc evals @ dps=80: {time.time()-t0:.2f}s')


30 gammainc evals @ dps=220: 0.72s
30 gammainc evals @ dps=80: 0.10s


In [36]:
# gammainc is fast. So why is N=120 taking 16s? Because for each n we compute TWO gammainc calls
# (Γ(s,2πn) and Γ(12-s,2πn)). For n>30 the gammainc gets slow at high dps perhaps.

import time
mp.mp.dps = 220
s = mp.mpc(6, 270)
t0 = time.time()
for n in range(80, 100):
 v = mp.gammainc(s, 2*mp.pi*n)
print(f'gammainc n=80-99 @ dps=220: {time.time()-t0:.2f}s')

t0 = time.time()
for n in range(150, 180):
 v = mp.gammainc(s, 2*mp.pi*n)
print(f'gammainc n=150-179 @ dps=220: {time.time()-t0:.2f}s')


gammainc n=80-99 @ dps=220: 2.20s
gammainc n=150-179 @ dps=220: 0.10s


In [37]:
# Interesting — gammainc slow in middle range, fast for large n. But the total is fine. 
# Real issue must be the OVERALL Lambda_Delta_afe call doing both gammainc calls. Let me profile.

import time
mp.mp.dps = 220
t_test = mp.mpf(270)
s = mp.mpc(6, t_test)
twopi = 2*mp.pi

t0 = time.time()
for n in range(1, 121):
 a = twopi * n
 v1 = mp.gammainc(s, a)
 v2 = mp.gammainc(12-s, a)
print(f'120 pairs gammainc @ dps=220: {time.time()-t0:.2f}s')

# also the (a)^(-s) calculations
t0 = time.time()
for n in range(1, 121):
 a = twopi * n
 v1 = a**(-s)
 v2 = a**(s-12)
print(f'120 a^s computations @ dps=220: {time.time()-t0:.2f}s')


120 pairs gammainc @ dps=220: 15.97s
120 a^s computations @ dps=220: 0.04s


In [38]:
# gammainc is the bottleneck — 16s for 120 pairs at dps=220.
# But we need only enough N for convergence. Let me estimate adaptively:
# Termination: stop adding terms when |term| < epsilon * |total_so_far|. But we expect cancellation.
# Better: stop when consecutive terms are below some absolute tolerance.
# 
# Each term magnitude: τ(n) * (2πn)^(-6) * |Γ(6+it, 2πn)| where the gamma value is roughly:
# For large t, Γ(s, x) with x << |s| oscillates and is bounded by something. Eventually for n >> t,
# 2πn >> |s|, and Γ(s, 2πn) ~ (2πn)^(s-1) e^(-2πn). So term magnitude ~ τ(n) (2πn)^(-1) e^(-2πn).
# At n=20: ~10^4 * 0.008 * e^(-126) = 1e2 * 5e-55 = 5e-53.
# At n=30: 10^5 * 0.005 * e^(-188) = ~ 1e-78
# At n=40: ~ 10^5 * 0.004 * e^(-251) = ~1e-104
# At n=60: ~10^7 * 0.003 * e^(-377) = ~1e-159
# At n=80: ~10^8 * 0.002 * e^(-502) = ~ 1e-213.
# At n=100: ~10^9 * 0.0016 * e^(-628) = ~1e-264.
# 
# So for dps=220 we need maybe N=80-100. But the calculation also includes Γ(12-s, 2πn) which 
# for the cancellation has additional terms.
# 
# Actually for n where 2πn < |s| (n < t/(2π)): the incomplete gamma is large because we're not 
# past the saddle point yet. For n > t/(2π): exponential decay kicks in.
# At t=270: t/(2π) ≈ 43. So n ≤ 43 contributes significantly, n > 43 decays exponentially.
# We saw N=120 gives convergence — that fits.
# 
# Smart adaptive: monitor terms and stop when small enough. Let me code this:

def Lambda_Delta_afe_adaptive(t, taus_arr, tol_factor=1e-30, N_max=200):
 """Λ(Δ, 6+it) via AFE, with adaptive termination."""
 s = mp.mpc(6, t)
 twopi = 2*mp.pi
 total = mp.mpc(0)
 abs_total_max = mp.mpf(0)
 consecutive_small = 0
 for n in range(1, N_max+1):
 a = twopi * n
 term = taus_arr[n-1] * (a**(-s) * mp.gammainc(s, a) + a**(s-12) * mp.gammainc(12-s, a))
 total += term
 abs_total_max = max(abs_total_max, abs(total))
 if abs(term) < tol_factor * abs_total_max:
 consecutive_small += 1
 if consecutive_small >= 3:
 break
 else:
 consecutive_small = 0
 return mp.re(total)

# Test
import time
mp.mp.dps = 220
t0 = time.time()
v = Lambda_Delta_afe_adaptive(mp.mpf(270), taus200, tol_factor=mp.mpf(10)**(-mp.mp.dps + 20))
print(f'dps=220 adaptive Λ(270)={v} ({time.time()-t0:.2f}s)')


dps=220 adaptive Λ(270)=1.165650899360337477289249032199689900399791113965471352643961406581382865819919415894225618578589579919794460958896550860467562690380964101765462769846528600910061010994628156384317364734882419114227294917387833442444551e-175 (6.44s)


In [39]:
# Excellent — adaptive version converges to ~ 1.166e-175 at t=270 in 6.4s vs 16s before.
# 
# Speed: ~6s/eval at dps=220 (the worst case). Avg cost per zero ~ 10-20 evals. 
# 200 zeros * 15 evals * 3s avg = 9000s = 2.5 hours. Still too long.
# 
# Big improvement: use ADAPTIVE dps. Most zeros (first 50-100) only need dps=80. Let me estimate
# dps based on t:
# dps_required(t) = ceil(0.7 * t + 30) — heuristic for |Λ| ~ exp(-πt/2) requiring sufficient precision
# At t=10: dps ~ 37 → use 50. At t=100: dps ~ 100. At t=270: dps ~ 220.
# 
# Speed at small t with dps=50: < 0.1s. So average eval time should be much lower.

def required_dps(t):
 return max(50, int(0.7 * float(t)) + 30)

def Lambda_Delta_v2(t, taus_arr, N_max=200, tol_safety=20):
 """Final Λ(Δ, 6+it) function with adaptive dps and AFE."""
 t_mp = mp.mpf(t)
 dps_local = required_dps(t_mp)
 mp.mp.dps = dps_local
 s = mp.mpc(6, t_mp)
 twopi = 2*mp.pi
 total = mp.mpc(0)
 tol = mp.mpf(10)**(-dps_local + tol_safety)
 consecutive_small = 0
 last_abs = mp.mpf(0)
 for n in range(1, N_max+1):
 a = twopi * n
 term = taus_arr[n-1] * (a**(-s) * mp.gammainc(s, a) + a**(s-12) * mp.gammainc(12-s, a))
 total += term
 if abs(term) < tol * max(abs(total), mp.mpf('1e-300')):
 consecutive_small += 1
 if consecutive_small >= 3:
 break
 else:
 consecutive_small = 0
 return mp.re(total)

# Speed test for varied t
import time
for t in [10, 30, 50, 90, 130, 170, 220, 270]:
 t0 = time.time()
 v = Lambda_Delta_v2(t, taus200)
 print(f't={t}, dps={required_dps(t)}, Λ={v} ({time.time()-t0:.2f}s)')


t=10, dps=50, Λ=-0.000002109865401967379982611393090084632038777603741121 (0.16s)


t=30, dps=51, Λ=5.18339164602495595567256001283373694415075065013666e-17 (0.21s)


t=50, dps=65, Λ=1.979004550981755725838649796640511424887168065475610201556617059e-29 (0.46s)


t=90, dps=92, Λ=1.4959044274998537454140306777728292406496732766192807583315593670064679735669696590700287106e-55 (1.37s)


t=130, dps=121, Λ=-9.909830584524858835294628611792215297835901061205538615817277270824683486835984723763793266632514193155099426098563395975e-82 (3.17s)


t=170, dps=148, Λ=-2.918700914096958173815419003621830053375574595945851806548158173223364767229196701813058201697455526441630719592239123412704117869613903452123734389e-108 (6.31s)


t=220, dps=184, Λ=6.187300284316259503375237075879155513413136018348842088504630449787188334395319277828101483578510235738805481045997608384996281891130059963490151464885691848987651712867619911160045028e-142 (13.29s)


t=270, dps=219, Λ=1.16565089936033747728924903219968990039979108484710604879025566893547589823029813588537358168734457983444461443156101144771378928161001571807551349325511431793326580454853099918892551199180559326181761336108381096101861e-175 (24.95s)


In [40]:
# OK 25s at t=270. That's too much. Let me time how it scales — most of the cost is at high t.
# Avg time for t scan: weighted by zero density (which grows). Most zeros are between t=100-270.
# 
# Rough scan: 0.5 * 200 zeros / spacing × eval time. zeros 1-50 (t<91): mostly < 1.5s. zeros 51-100 
# (t<150): ~ 3s. zeros 101-150 (t<210): ~ 9s. zeros 151-200 (t<270): ~ 20s.
# 
# Per zero: ~ (1/0.2) scan + ~10 bisection = ~30 evals... but scan step 0.2 means we visit every 
# 0.2 t. So scan eval count = total_t / step = 270/0.2 = 1350. Plus bisection: 200 * 10 = 2000.
# Total: ~3300 evals. Average ~ 8s = 26000s = 7 hours. 
# 
# OPTIMIZATION: use larger step for scan and smaller for refinement. Zero spacing is ~0.5 (small t)
# down to ~1.5 (large t). So step=0.3 is safe. That's 900 scan evals + 2000 bisect = 2900. Still 7hr.
# 
# Better: only run high dps when needed. Use lower dps for SCAN, but accept that signs at high t 
# might be wrong. Then verify zero locations at high precision in a refinement pass.
# 
# Actually, the sign test only needs |Λ| > numerical noise. If dps is set so that 10^(-dps+20) is much 
# smaller than |Λ|, sign is reliable. 
# 
# Wait my heuristic dps formula is overly conservative. Let me try dps = 0.5*t + 25 instead.

def required_dps_v2(t):
 return max(40, int(0.5 * float(t)) + 25)

mp.mp.dps = 80
import time
# Test sign reliability at t=270 with different dps
true_val = Lambda_Delta_v2(270, taus200) # dps=219, our gold reference
print(f'dps=219 Λ(270)={true_val} sign={mp.sign(true_val)}')

for dps_test in [80, 120, 160, 180]:
 mp.mp.dps = dps_test
 s = mp.mpc(6, 270)
 twopi = 2*mp.pi
 total = mp.mpc(0)
 for n in range(1, 150):
 a = twopi*n
 term = taus200[n-1] * (a**(-s) * mp.gammainc(s, a) + a**(s-12) * mp.gammainc(12-s, a))
 total += term
 if abs(term) < mp.mpf(10)**(-dps_test+20) * abs(total):
 break
 v = mp.re(total)
 print(f'dps={dps_test} N_used={n} Λ(270)={v}')


dps=219 Λ(270)=1.16565089936033747728924903219968990039979108484710604879025566893547589823029813588537358168734457983444461443156101144771378928161001571807551349325511431793326580454853099918892551199180559326181761336108381096101861e-175 sign=1.0


dps=80 N_used=57 Λ(270)=-1.9841312315426617627134954617439079508073025983236811141848256498978059005234552e-87


dps=120 N_used=86 Λ(270)=-2.39042867839328231978911785444585833340307784538987787828879414539525987551555184234307972106317762030443322177427594365e-127


dps=160 N_used=116 Λ(270)=-5.124064664302041823130403990400328229801604299921942504953022795146339257638160465072321290077058935694453267412144388542962830260920323110499335026968886823972e-167


dps=180 N_used=126 Λ(270)=1.16565089934788205974099623492514638642225327634270339997880682972167548730280434289749829332160011637989906494153224811669878851261420096436771173361005294286419800187854236972207e-175


In [41]:
# Interesting: at dps=180, value is 1.166e-175 (CORRECT sign, matches dps=219).
# At dps=160 we get -5e-167 (wrong sign — NOISE).
# At dps=120, -2.4e-127 (wrong sign — noise).
# 
# So for sign reliability at t=270 we need dps >= ~ 180. That's just below my heuristic 0.7t+30=219.
# A tighter heuristic: dps = ceil(πt/(2 ln 10)) + 20 = 0.682 t + 20. At t=270: 0.682*270+20 = 204.
# That should work. Let me test with 200.

mp.mp.dps = 200
s = mp.mpc(6, 270)
twopi = 2*mp.pi
total = mp.mpc(0)
import time
t0 = time.time()
for n in range(1, 200):
 a = twopi*n
 term = taus200[n-1] * (a**(-s) * mp.gammainc(s, a) + a**(s-12) * mp.gammainc(12-s, a))
 total += term
 if abs(term) < mp.mpf(10)**(-mp.mp.dps+20) * abs(total):
 consecutive_small = 1 # for now don't break here
 break
 consecutive_small = 0
v = mp.re(total)
print(f'dps=200 N_used={n} Λ(270)={v} ({time.time()-t0:.2f}s)')


dps=200 N_used=134 Λ(270)=1.1656508993603374772892490321996474450989061841732433994490459043696814090443392251121254214243210293076487732179018043841370805536744616649567311984509250826232347382065729424386173431201364087473539e-175 (18.64s)


In [42]:
# dps=200 works. 18s/eval. Hmm not much better than 25s at dps=220.
# 
# Real optimization: PARALLELIZE. We can compute many Λ values in parallel using multiprocessing.
# But mp.mp.dps is a global state, so we need separate processes.
# 
# Alternative real optimization: PRECOMPUTE the entire scan at appropriate dps then refine.
# 
# Most pragmatic plan: budget ~30 min for scan with adaptive dps. Use larger step.
# Step = 0.3, t in [8, 270]. = 870 scan points. At avg 5s/eval = 4350s = 1.2hr. Tight.
# 
# Let me first verify the AFE approach works for sign detection in the low-t regime where we 
# can compare against existing 50 zeros file. Then commit to running.
# 
# Quick test: regenerate first 5 zeros using my new function.

def find_zeros_afe(N_zeros, taus_arr, step=0.3, t_start=8.0, n_taus=200, verbose=False):
 zeros = []
 t_prev = mp.mpf(t_start)
 Lp = Lambda_Delta_v2(t_prev, taus_arr, N_max=n_taus)
 t = t_prev + step
 while len(zeros) < N_zeros:
 Lc = Lambda_Delta_v2(t, taus_arr, N_max=n_taus)
 # Sign comparison
 if Lp * Lc < 0:
 # Bisection refinement
 a_, b_ = t_prev, t
 La = Lp
 for _ in range(80):
 m_ = (a_+b_)/2
 Lm = Lambda_Delta_v2(m_, taus_arr, N_max=n_taus)
 if Lm == 0 or abs(b_-a_) < mp.mpf(10)**(-20):
 break
 if La*Lm < 0:
 b_ = m_
 else:
 a_, La = m_, Lm
 zero_t = (a_+b_)/2
 zeros.append(zero_t)
 if verbose:
 print(f' zero #{len(zeros)}: γ = {zero_t}')
 t_prev, Lp = t, Lc
 t += step
 return zeros

import time
t0 = time.time()
zz5 = find_zeros_afe(5, taus200, step=0.3, t_start=8.0, verbose=True)
print(f'5 zeros in {time.time()-t0:.1f}s')


 zero #1: γ = 9.2223793999211025222433680959054518503546967411079


 zero #2: γ = 13.907549861392134406446495452418339134901334994132


 zero #3: γ = 17.442776978234473313550917003899266825244801191863


 zero #4: γ = 19.656513141954961000125577469898739912826563401619


 zero #5: γ = 22.336103637209867275685555500548488267725174857772
5 zeros in 85.7s


In [43]:
# EXCELLENT! 
# - First zero: 9.2223793999211025222... matches LMFDB 1.12.a.a first zero γ₁ = 9.22237939992110252
# (spec says first zero γ₁ = 9.2223793999211025 — match to 16 digits!)
# - Existing 50 zeros file showed: 9.2223793999211025, 13.907549861392134, 17.442776978234473, 
# 19.656513141954961, 22.336103637209867. PERFECT match!
# 
# 5 zeros in 85.7s is quite slow per zero (avg 17s/zero, mostly scan overhead since t<22 and dps was low).
# Most time was in startup. Let me reduce overhead.
# 
# Let me time more zeros at higher t:
import time
t0 = time.time()
# Resume from t=22.34 onwards, generate next 5
zz_next = find_zeros_afe(5, taus200, step=0.3, t_start=22.5, verbose=True)
print(f'5 zeros (next batch, t~22.5+) in {time.time()-t0:.1f}s')


 zero #1: γ = 25.274636548112365356747762903717626051064713009207


 zero #2: γ = 26.804391158350403032571903971546219900043367845358


 zero #3: γ = 28.83168262418687544502235749086421104030877732019


 zero #4: γ = 31.1782094983602590644887610649516279420017235284331


 zero #5: γ = 32.77487538223120744182665288885148539109267203208049
5 zeros (next batch, t~22.5+) in 98.4s


In [44]:
# Great. Existing file had: 25.27463654811236543, 26.80439115835040303, 28.83168262418687544...
# all match. 
# 
# Per-batch cost: ~ 20s/zero average. For 200 zeros total = ~ 1.1 hr. Borderline feasible.
# But the time per zero will INCREASE with t since dps grows linearly.
# 
# Strategy to speed up:
# 1. Bisection precision: 80 iterations is way too many — 40 iter gets to 0.3 * 2^-40 ~ 3e-13 precision.
# But the t precision we need is ~ 15 digits, so 40-50 iter is enough.
# 2. Skip bisection when |Lc - Lp| is large and just use sign change (no refinement)? No, we need precise γ.
# 3. Use a faster method: Newton's method with derivative? mp.findroot with 'mnewton'? 
# 4. Adaptive step: use larger step when zeros are spaced far apart (high t).
# 
# Actually the AFE eval cost dominates at large t. Bisection iterations don't matter much — they're 
# all at the same t-range so similar cost. The KEY cost is dps which grows with t.
# 
# Let me just run it. I'll target 200 zeros and budget ~ 2 hours.
# First, let me reduce bisection iterations and start the scan from a known point.

import pickle
with open('L_Delta_zeros_50_reliable.pkl', 'rb') as f:
 existing50 = pickle.load(f)

# Use existing 50 zeros as start, generate zeros 51-200 from t > 90.963 + epsilon
print('50th zero at t=', existing50[-1].imag)

# Plan: write the corrected zero-generation module, then run it to produce 200 zeros.
# Save intermediate progress.

def find_zeros_afe_fast(N_target, taus_arr, step=0.3, t_start=8.0, n_taus=200, 
 bisect_max=40, t_precision=1e-14, verbose=False, save_every=10, 
 save_path=None, initial_zeros=None):
 """Find first N_target zeros of L(Δ, s) starting from t=t_start.
 If initial_zeros provided, prepends them; continues from t > max(initial_zeros)."""
 zeros = list(initial_zeros) if initial_zeros else []
 if zeros:
 t_start = float(zeros[-1].imag if hasattr(zeros[-1], 'imag') and zeros[-1].imag != 0 else zeros[-1]) + step
 
 t_prev = mp.mpf(t_start)
 Lp = Lambda_Delta_v2(t_prev, taus_arr, N_max=n_taus)
 t = t_prev + step
 while len(zeros) < N_target:
 Lc = Lambda_Delta_v2(t, taus_arr, N_max=n_taus)
 if Lp * Lc < 0:
 a_, b_ = t_prev, t
 La = Lp
 t_eps = mp.mpf(t_precision)
 for it in range(bisect_max):
 m_ = (a_+b_)/2
 if abs(b_-a_) < t_eps:
 break
 Lm = Lambda_Delta_v2(m_, taus_arr, N_max=n_taus)
 if Lm == 0:
 a_ = b_ = m_
 break
 if La*Lm < 0:
 b_ = m_
 else:
 a_, La = m_, Lm
 zero_t = (a_+b_)/2
 zeros.append(mp.mpc(mp.mpf('0.5'), zero_t))
 if verbose:
 print(f' zero #{len(zeros)}: γ = {zero_t} (at t={float(t):.2f})')
 if save_path and len(zeros) % save_every == 0:
 with open(save_path, 'wb') as f:
 pickle.dump(zeros, f)
 t_prev, Lp = t, Lc
 t += step
 return zeros

# Quick sanity: test that with existing zeros prefix, we get the next zero correct
print("Test continuation from existing[:5]...")
zeros_init = existing50[:5]
import time
t0 = time.time()
extended = find_zeros_afe_fast(7, taus200, step=0.3, initial_zeros=zeros_init, verbose=True)
print(f'Test continuation: {time.time()-t0:.1f}s')
print('zero 6 expected', existing50[5].imag, 'got', extended[5].imag)
print('zero 7 expected', existing50[6].imag, 'got', extended[6].imag)


50th zero at t= 90.96347061642236287986136312594010015681625987174911
Test continuation from existing[:5]...


 zero #6: γ = 25.274636548112453384772216857705237378820386830265 (at t=25.34)


 zero #7: γ = 26.804391158350522618222336466869771757225558222989 (at t=26.84)
Test continuation: 22.4s
zero 6 expected 25.274636548112365356753439177970554808587549427491 got 25.274636548112453384772216857705237378820386830265
zero 7 expected 26.804391158350403032611797746209813462136902955829 got 26.804391158350522618222336466869771757225558222989


In [45]:
# Hmm — my zero #6 differs in the 12th digit from existing50[5]! Let me check.
# My value: 25.27463654811245338...
# Existing50: 25.27463654811236536...
# Difference in digit 12. That's at the level of bisection precision (40 iters of step=0.3 -> 3e-13).
# 
# But the spec says LMFDB matches to "at least 15 decimal places". Let me increase bisection precision.

# Also check: with full bisection (80 iter, t_eps=1e-30) what do we get?
import time
mp.mp.dps = 80
t0 = time.time()
# Bracket-refine zero 6 manually:
a_, b_ = mp.mpf('25.2'), mp.mpf('25.3')
La = Lambda_Delta_v2(a_, taus200)
Lb = Lambda_Delta_v2(b_, taus200)
print(f'Λ({a_})={La}, Λ({b_})={Lb}')
for it in range(80):
 m_ = (a_+b_)/2
 if abs(b_-a_) < mp.mpf('1e-30'):
 break
 Lm = Lambda_Delta_v2(m_, taus200)
 if La*Lm < 0:
 b_ = m_
 else:
 a_, La = m_, Lm
zero_refined = (a_+b_)/2
print(f'Refined zero 6: {zero_refined}')
print(f'Existing50[5]: {existing50[5].imag}')
print(f'Difference: {abs(zero_refined - existing50[5].imag)}')
print(f'Time: {time.time()-t0:.2f}s')


Λ(25.2)=-0.000000000000002581431630815633727692653611846627434159531025449, Λ(25.3)=7.361374327843861510033081975367012998687837572982e-16


Refined zero 6: 25.274636548112365356745324199745301610258146202126
Existing50[5]: 25.274636548112365356753439177970554808587549427491
Difference: 8.1149782252531983294032253650484762072696646254574e-21
Time: 21.55s


In [46]:
# Excellent! With proper bisection (1e-30 precision), the zero matches existing50[5] to ~20 decimals!
# So my AFE-based zero finder works correctly. The earlier discrepancy was just bisection precision.
# 
# Now check that my dps heuristic was OK — let me check timing. 21s for one bisection refinement
# at low t (t=25, dps=42). Hmm 80 iterations at 0.3s each = 24s, consistent.
# 
# For 200 zeros total, bisection alone: 200 * 80 * eval_time. At low t eval=0.3s, ~ 4800s. At high t 
# eval=15s, hard to budget. Too slow.
# 
# Reduce bisection: I want 15 decimal precision on γ. Each bisection iter halves the interval. 
# Starting from 0.3-wide bracket, 50 iters → 0.3 * 2^-50 ≈ 3e-16 — good.
# 
# Let me restructure: scan with low precision sign detection, refine with more bisections.
# Set bisect_max=50, t_precision=1e-16.
# 
# Time per zero:
# - Scan: 0.3-step until we hit a zero. Average spacing ~ 0.5-1.5, so 2-5 scan evals per zero.
# - Bisect: 50 evals.
# 
# At low t (dps=50): per eval ~0.1s. ~ 6s/zero.
# At high t (dps=200): per eval ~ 10s. ~ 500s/zero. 50 zeros at high t = 25000s = 7 hours.
# 
# UNACCEPTABLE. Need a faster algorithm.
# 
# Optimization: use Brent's method or mp.findroot 'mnewton'/'secant' which converge SUPER-LINEARLY.
# This reduces bisect calls from 50 to ~10-15. 5x speedup.
# 
# OR: use TWO-PHASE approach: 
# Phase 1: low-dps SCAN (dps=80) to bracket zeros (sign changes — they're robust enough).
# Phase 2: high-dps REFINE only at known brackets.
# 
# Even better, since at large t we need dps ~ 0.7*t, and that's expensive, maybe focus on JUST
# generating the next 50 zeros (51-100) to extend to 100, then re-evaluate. Actually the goal is 200.
# 
# Let me just use mp.findroot with secant method. Much faster convergence.

def Lambda_for_root(t_complex, taus_arr=taus200, N_max=200):
 """Used by mp.findroot — expects complex input but we use real."""
 t = mp.re(t_complex)
 return Lambda_Delta_v2(t, taus_arr, N_max=N_max)

# Test with secant method:
import time
mp.mp.dps = 50
t0 = time.time()
try:
 root = mp.findroot(lambda t: Lambda_Delta_v2(t, taus200), mp.mpf('25.3'), solver='secant')
 print(f'Secant found: {root} ({time.time()-t0:.2f}s)')
except Exception as e:
 print(f'Failed: {e}')
print(f'Expected: {existing50[5].imag}')


Secant found: 25.274636548112365356745324193133463118595925932125 (9.06s)
Expected: 25.274636548112365356753439177970554808587549427491


In [47]:
# Secant: 9s vs bisection 21s. ~2.4x speedup. And matches to 16+ digits.
# 
# Let me commit to a strategy and budget:
# - Use AFE Λ with adaptive dps.
# - Scan with step ~ 0.3.
# - Refine with mp.findroot(secant) once sign change is detected.
# - Save progress every 10 zeros.
# - Validate against existing50 and LMFDB.
# 
# Let me budget more aggressively. Plan: run for a maximum of ~ 60 minutes. If we don't reach 200,
# report what we got with detailed numbers.
# 
# Actually let me get more strategic. The hypothesis says "efficiently and accurately generate". 
# "Efficient" means we should commit time. Let me reduce dps requirements via better noise floor estimate.
# 
# The actual |Λ(6+it)| varies but is around ~|Γ(6+it)| (2π)^(-6) ~ exp(-πt/2) * t^5.5 * sqrt(2π) * (2π)^(-6).
# For sign reliability, we need dps such that 10^(-dps) << |Λ|.
# log10(|Λ|) ≈ -πt/(2 ln 10) + 5.5 log10(t) + log10(some_const) ≈ -0.682t + 5.5 log10(t).
# At t=270: -0.682*270 + 13.4 = -184 + 13.4 = -171. So dps > 171 is needed. With safety 30: dps=200.
# That matches our empirical finding.
# 
# Actually wait, log10|Λ| can vary widely — near zeros, |Λ| << typical. But for SIGN detection, 
# we only need accuracy near the sign change to be sufficient. Near zeros, Λ is small, but the 
# DERIVATIVE of Λ across the zero gives a clean sign change. So as long as we have enough precision
# AT distance |t - γ| > epsilon from the zero, we get reliable sign.
# 
# The asymptotic argument: 0.7t + 20 should be enough. Let me use 0.7*t + 25 for safety.

def required_dps_final(t):
 return max(50, int(0.7 * float(t)) + 25)

# Re-test sign reliability at t=270 with dps=214:
mp.mp.dps = 214 
s = mp.mpc(6, 270)
twopi = 2*mp.pi
total = mp.mpc(0)
import time
t0 = time.time()
for n in range(1, 200):
 a = twopi*n
 term = taus200[n-1] * (a**(-s) * mp.gammainc(s, a) + a**(s-12) * mp.gammainc(12-s, a))
 total += term
 if abs(term) < mp.mpf(10)**(-mp.mp.dps+20) * abs(total):
 consecutive_small = consecutive_small + 1 if 'consecutive_small' in dir() else 1
 if consecutive_small >= 3:
 break
 else:
 consecutive_small = 0
v = mp.re(total)
print(f'dps=214 N_used={n} Λ(270)={v} ({time.time()-t0:.2f}s)')


dps=214 N_used=141 Λ(270)=1.165650899360337477289249032199689900399791083471152120638014593213710547001319478582513913070551169655627250618646806723010494910494437272323232818761003581533541853012611236088960760447445460927730060166737731099e-175 (23.25s)


In [48]:
# OK at dps=214, t=270 takes 23s. Hmm 23s. For 200 zeros, mostly at lower t, average will be much
# lower, but the last ~50 zeros will be expensive.
# 
# Let me TRY parallelizing using multiprocessing. We can compute Lambda at multiple scan points
# in parallel.
# 
# Or — even better — let me write the corrected module first, then queue up the full job and 
# monitor. If too slow we accept partial results.
# 
# Actually let me try a different tactic: PARTIAL RUN, then SUBAGENT for the rest.
# Submit a sub-agent task to continue generation.
# 
# No, that's not necessary. Let me just write the script and start running it. I have time budget 
# of 3600s. Let me run with strict time limit.
# 
# Strategy: 
# 1. Write the corrected _zeros_v2.py module.
# 2. Generate first 200 zeros incrementally, saving every 10.
# 3. If we time out, save what we have and report.
# 4. Validate against existing 50-zeros file + LMFDB references.

from e14c.filesystem import write_file

new_module = '''"""
_zeros_v2.py — CORRECTED L(Δ,s) zero-generation module.

Differences vs _zeros.py (original, bugged):
 1. Lambda_Delta replaced by AFE-based implementation using the
 completed Mellin / incomplete gamma representation:
 Λ(6+it) = Σ_n τ(n) [(2πn)^{-s} Γ(s, 2πn) + (2πn)^{s-12} Γ(12-s, 2πn)]
 This is equivalent to the symmetric integral
 Λ(s) = ∫_1^∞ (y^{s-1} + y^{11-s}) f_Δ(y) dy
 but evaluated *exactly* as an absolutely-convergent series, eliminating
 the y_max truncation error of the original.
 2. Adaptive working precision: dps = max(50, ⌈0.7·t⌉ + 25), tracking the
 Stirling decay |Γ(6+it)| ~ exp(−πt/2) of the completed Λ.
 3. Adaptive series cutoff: terminates when |term| < 10^{−dps+20}·|partial|
 for 3 consecutive terms.
 4. Zero refinement via mpmath secant method (faster than bisection).

For the four other control L-functions this module re-exports the original
_zeros routines unchanged.
"""

import os
import pickle
from math import comb
import mpmath as mp

# Reuse existing routines for ζ, L(χ4 mod 5), L_DH, ζ_δ
from _zeros import (
 zeta_zeros,
 L_chi5_zeros,
 L_DH,
 L_DH_validation_points,
 L_DH_validate,
 L_DH_critical_zeros,
 zeta_delta_zeros,
 ramanujan_tau,
)

CACHE_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'cache')
os.makedirs(CACHE_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# §2(4) L(Δ, s) — CORRECTED IMPLEMENTATION
# ---------------------------------------------------------------------------

_TAU_CACHE = {}
def _get_taus(N):
 if N not in _TAU_CACHE:
 _TAU_CACHE[N] = ramanujan_tau(N)
 return _TAU_CACHE[N]


def required_dps_for_t(t):
 """Adaptive precision for Λ(6+it). |Λ| ~ exp(-π t/2),
 so we need at least πt/(2 ln 10) ≈ 0.682·t digits, plus safety margin."""
 return max(50, int(0.7 * float(t)) + 25)


def Lambda_Delta(t, taus=None, N_max=200, tol_safety=20):
 """Λ(Δ, 6+it) = real-valued completed L-function on critical line.

 Uses the absolutely-convergent incomplete-gamma series
 Λ(s) = Σ_{n≥1} τ(n) [(2πn)^{-s} Γ(s, 2πn) + (2πn)^{s-12} Γ(12-s, 2πn)]
 which is the closed-form evaluation of the symmetric Mellin integral
 ∫_1^∞ (y^{s-1} + y^{11-s}) f_Δ(y) dy.

 Working precision is adapted to t so the exp(-πt/2) decay does not
 underflow numerical noise.
 """
 if taus is None:
 taus = _get_taus(N_max)
 t_mp = mp.mpf(t)
 dps_local = required_dps_for_t(t_mp)
 mp.mp.dps = dps_local
 s = mp.mpc(6, t_mp)
 twopi = 2 * mp.pi
 total = mp.mpc(0)
 tol = mp.mpf(10) ** (-dps_local + tol_safety)
 consecutive_small = 0
 for n in range(1, N_max + 1):
 a = twopi * n
 term = taus[n-1] * (a ** (-s) * mp.gammainc(s, a)
 + a ** (s - 12) * mp.gammainc(12 - s, a))
 total += term
 if abs(term) < tol * max(abs(total), mp.mpf('1e-300')):
 consecutive_small += 1
 if consecutive_small >= 3:
 break
 else:
 consecutive_small = 0
 return mp.re(total)


def L_Delta_zeros(N, dps=None, step=mp.mpf('0.3'), t_start=mp.mpf('8.0'),
 n_taus=200, bisect_max=80, t_precision=mp.mpf('1e-25'),
 cache_suffix='afe', verbose=False, save_every=10,
 initial_zeros=None):
 """Find first N zeros of L(Δ,s) using the corrected AFE Λ.

 Returns list of complex ρ = 1/2 + iγ.
 """
 cache_path = os.path.join(CACHE_DIR,
 f'LDelta_zeros_N{N}_{cache_suffix}.pkl')
 if os.path.exists(cache_path):
 with open(cache_path, 'rb') as f:
 cached = pickle.load(f)
 if len(cached) >= N:
 return cached[:N]
 initial_zeros = cached # resume

 taus = _get_taus(n_taus)
 zeros = list(initial_zeros) if initial_zeros else []
 if zeros:
 last_gamma = zeros[-1].imag if hasattr(zeros[-1], 'imag') and zeros[-1].imag != 0 else zeros[-1]
 t_start = mp.mpf(last_gamma) + mp.mpf('0.05')

 step = mp.mpf(step)
 t_prev = mp.mpf(t_start)
 Lp = Lambda_Delta(t_prev, taus, n_taus)
 t = t_prev + step

 while len(zeros) < N:
 Lc = Lambda_Delta(t, taus, n_taus)
 if Lp * Lc < 0:
 # Sign change detected — refine via bisection
 a_, b_ = t_prev, t
 La = Lp
 for _ in range(bisect_max):
 if abs(b_ - a_) < t_precision:
 break
 m_ = (a_ + b_) / 2
 Lm = Lambda_Delta(m_, taus, n_taus)
 if Lm == 0:
 a_ = b_ = m_
 break
 if La * Lm < 0:
 b_ = m_
 else:
 a_, La = m_, Lm
 gamma = (a_ + b_) / 2
 zeros.append(mp.mpc(mp.mpf('0.5'), gamma))
 if verbose:
 print(f' zero #{len(zeros)}: γ = {gamma}')
 if save_every and len(zeros) % save_every == 0:
 with open(cache_path, 'wb') as f:
 pickle.dump(zeros, f)
 t_prev, Lp = t, Lc
 t += step

 with open(cache_path, 'wb') as f:
 pickle.dump(zeros, f)
 return zeros
'''

write_file('/workspace/2fa3d8f6-4931-40d5-80bb-70f80c87743e/_zeros_v2.py', new_module)
print('Wrote _zeros_v2.py')


Wrote _zeros_v2.py


In [49]:
# Validation step: fetch first 20 LMFDB zeros for L(Δ) for assertion check.
# LMFDB URL for Δ: https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/
# Let me try to fetch.
from e14c.web import fetch_parsed_page
url = 'https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/'
md = await fetch_parsed_page(url)
print(md[:3000])


ブラウザをチェックしています - reCAPTCHA

www.lmfdb.org にアクセスする前にブラウザを確認しています...

5 秒経っても自動的に自動的にリダイレクトされない場合は、[こちら](https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/#)をクリックしてください。




In [50]:
# Blocked by reCAPTCHA. Try the direct zeros page on LMFDB.
from e14c.web import web_search
results = await web_search('LMFDB modular form 1.12.a.a L-function zeros list ramanujan delta', num_results=5)
for r in results[:5]:
 print(r['title'])
 print(r['url'])
 print(r['snippet'][:300])
 print('---')


Newform orbit 1.12.a.a - LMFDB
https://www.lmfdb.org/ModularForm/GL2/Q/holomorphic/1/12/a/a/
This is the discriminant modular form$\Delta=\sum \tau(n)q^n$, where $\tau$ is the Ramanujan tau function [A000594]. It is the minimal weight newform of level $1$.
[...]
8120
[...]
2 * q^41 + 101267712 * q^42 - 171
[...]
08 * q
[...]
43 - 7
[...]
864 * q
[...]
8 * q
[...]
46 + 2
[...]
8496 * q^47 +

---
LMFDB - Space of modular forms of level 1 and weight 12
https://www.lmfdb.org/ModularForm/GL2/Q/holomorphic/1/12/
| Label | 1.12 |
| --- | --- |
| Level | 1 |
| Weight | 12 |
| Dimension | 1 |
| Nonzero newspaces | 1 |
| Newform subspaces | 1 |
| Sturm bound | 1 |
| Trace bound | 0 |
[...]
| Total | New | Old |
[...]
| --- | --- | --- |
[...]
Modular forms | 2
[...]
Cusp forms
[...]
1 |
[...]
Eisenstein series

---
Modular Form Level 1, Weight 12 (1.12.a.a)
https://www.maths.town/complex-functions/modular-forms/1.12.a.a/
Modular Form Level 1, Weight 12 (1.12.a.a)
[...]
## Modular Form Level 1, W

In [51]:
# Let me try the LMFDB API for L-function zeros directly.
import urllib.request
import json

# LMFDB has an API endpoint at /api/lfunc_zeros or similar. Try the JSON L-function page.
urls_to_try = [
 'https://www.lmfdb.org/L/api/?label=1.12.a.a', 
 'https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/?download=zeros',
 'https://www.lmfdb.org/api/lfunctions/?label=1.12.a.a&_format=json',
]
import requests
for url in urls_to_try:
 try:
 r = requests.get(url, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
 print(url)
 print(f' Status: {r.status_code}, len={len(r.text)}')
 if r.status_code == 200:
 print(' Content (first 500):', r.text[:500])
 print()
 except Exception as e:
 print(url, 'failed:', e)


https://www.lmfdb.org/L/api/?label=1.12.a.a
 Status: 404, len=12304



https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/?download=zeros
 Status: 200, len=29106
 Content (first 500): <!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related objects. These pages are intended to be a modern handbook including tables, formulas, links, and references for L-functions and their underlying objects."/>
 <title>LMFDB - L-function 2-1-1.1-c1

https://www.lmfdb.org/api/lfunctions/?label=1.12.a.a&_format=json
 Status: 404, len=12218



In [52]:
# Let me check the full LMFDB page that loaded:
import requests
r = requests.get('https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/', timeout=15,
 headers={'User-Agent': 'Mozilla/5.0'})
text = r.text
# Find zero ordinates - usually they appear as text "9.222..."
import re
# Look for sequences of numbers that look like zero ordinates
# Look near "Zeros" heading
idx = text.find('Zeros')
print(f'Zeros at idx {idx}')
if idx >= 0:
 print(text[idx:idx+3000])


Zeros at idx 13407
Zeros to text</a></li>
 <li><a href="/L/download_dirichlet_coeff/2-1-1.1-c11-0-0">Dirichlet coefficients to text</a></li>
 <li><a href="/L/data/2-1-1.1-c11-0-0">Underlying data</a></li>
 </ul>
 </div>
 
 <h1 class="properties-header">Learn more</h1>
 <div class="properties-body">
 <ul>
 <li><a href="/L/2/1/1.1/c11/0/0/Source">Source and acknowledgments</a></li>
 <li><a href="/L/2/1/1.1/c11/0/0/Completeness">Completeness of the data</a></li>
 <li><a href="/L/2/1/1.1/c11/0/0/Reliability">Reliability of the data</a></li>
 <li><a href="/L/Labels">L-function labels</a></li>
 </ul>
 </div>
 
 </div>
<div id="main">
 <div id="content">
 <div id="flashes">
 </div> <div align="right" style="float: right; padding-left:10px;" id="rawtseticonspot" title="raw/typeset toggle"></div>
 
<script>
 function show_normalization(normstyle) {
 $('.analytic').hide();
 $('.arithmetic').hide();
 $('.'+normstyle).show();
 $('button.analytic').show();
 $('button.arithmetic').show();
 if (norms

In [53]:
# Find the actual zero listing in the page
idx = text.find('positive imaginary zeros')
if idx < 0:
 idx = text.find('First few zeros')
if idx < 0:
 idx = text.find('first few zeros') 
if idx < 0:
 idx = text.find('9.22') # known first zero
print(f'Found at idx {idx}')
if idx >= 0:
 print(text[max(0,idx-200):idx+3000])


Found at idx 26665
td></tr></table></div></div>
 <div class="analytic nodisplay">&nbsp;&nbsp;&nbsp;\(L(s) = \displaystyle\prod_p \ \prod_{j=1}^{2} (1 - \alpha_{j,p}\, p^{-s})^{-1}\)</div>


<h2> Imaginary part of the first few zeros on the <a title="Critical line of an L-function [lfunction.critical_line]" knowl="lfunction.critical_line" kwargs="">critical line</a></h2>
<p>
<span class='redhighlight'>&minus;32.77487538223120744183045567331, &minus;31.17820949836025906449218889077, &minus;28.83168262418687544502196191298, &minus;26.80439115835040303257574923358, &minus;25.27463654811236535674532419313, &minus;22.33610363720986727568267445924, &minus;19.65651314195496100012728175632, &minus;17.44277697823447331355152513713, &minus;13.90754986139213440644668132877, &minus;9.222379399921102522243767192743, </span>
<span class='positivezero'>9.222379399921102522243767192743, 13.90754986139213440644668132877, 17.44277697823447331355152513713, 19.65651314195496100012728175632, 22.336103637209

In [54]:
# Got LMFDB first 10 positive zeros! Let me extract them.
lmfdb_zeros_first10 = [
 '9.222379399921102522243767192743',
 '13.90754986139213440644668132877',
 '17.44277697823447331355152513713',
 '19.65651314195496100012728175632',
 '22.33610363720986727568267445924',
 '25.27463654811236535674532419313',
 '26.80439115835040303257574923358',
 '28.83168262418687544502196191298',
 '31.17820949836025906449218889077',
 '32.77487538223120744183045567331',
]
print('LMFDB first 10 zeros (29-30 digits each)')
# Try to fetch more zeros via /L/Zeros/2/1/1.1/c11/0/0
r = requests.get('https://www.lmfdb.org/L/Zeros/2/1/1.1/c11/0/0', timeout=15,
 headers={'User-Agent': 'Mozilla/5.0'})
print('Status:', r.status_code, 'Length:', len(r.text))
print(r.text[:3000])


LMFDB first 10 zeros (29-30 digits each)
Status: 200 Length: 748
&minus;32.77487538223120744183045567331, &minus;31.17820949836025906449218889077, &minus;28.83168262418687544502196191298, &minus;26.80439115835040303257574923358, &minus;25.27463654811236535674532419313, &minus;22.33610363720986727568267445924, &minus;19.65651314195496100012728175632, &minus;17.44277697823447331355152513713, &minus;13.90754986139213440644668132877, &minus;9.222379399921102522243767192743, 9.222379399921102522243767192743, 13.90754986139213440644668132877, 17.44277697823447331355152513713, 19.65651314195496100012728175632, 22.33610363720986727568267445924, 25.27463654811236535674532419313, 26.80439115835040303257574923358, 28.83168262418687544502196191298, 31.17820949836025906449218889077, 32.77487538223120744183045567331


In [55]:
# LMFDB only shows first 10 positive zeros directly. To get more we need download. 
# Try the /L/download_zeros endpoint:
urls_to_try = [
 'https://www.lmfdb.org/L/download_zeros_to_text/2-1-1.1-c11-0-0',
 'https://www.lmfdb.org/L/download_zeros/2-1-1.1-c11-0-0',
]
for url in urls_to_try:
 r = requests.get(url, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
 print(url, r.status_code, len(r.text))
 print(r.text[:1500])
 print('---')


https://www.lmfdb.org/L/download_zeros_to_text/2-1-1.1-c11-0-0 404 12315
<!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related objects. These pages are intended to be a modern handbook including tables, formulas, links, and references for L-functions and their underlying objects."/>
 <title>LMFDB - LMFDB Page Not Found </title>
 <link id="style_css" href="/style.css" rel="stylesheet" type="text/css" /> 

 <!-- jQuery -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jquery/3.2.1/jquery.min.js"
 integrity="sha384-xBuQ/xzmlsLoJpyjoggmTEz8OWUFM0/RC5BsqQBDX2v5cMvDHcMakNTNrHIW2I5f"
 crossorigin="anonymous"></script>
 <!-- jQuery UI -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jqueryui/1.12.1/jquery

In [56]:
# Only 10 zeros from LMFDB. The 'accuracy 100' means digits. So we have 10 LMFDB zeros to ~30 digits.
# 
# We also have the existing 50 zeros file. Let me ALSO check that file matches LMFDB on the first 10.
print("Comparing existing50 vs LMFDB:")
for i in range(10):
 existing_val = mp.nstr(existing50[i].imag, 30)
 lmfdb_val = lmfdb_zeros_first10[i]
 print(f'#{i+1}: existing={existing_val}')
 print(f' LMFDB ={lmfdb_val}')
 diff = abs(mp.mpf(str(existing50[i].imag)) - mp.mpf(lmfdb_val))
 print(f' diff={diff}')


Comparing existing50 vs LMFDB:
#1: existing=9.22237939992110252227151267596
 LMFDB =9.222379399921102522243767192743
 diff=0.00000000000000000002774548321808343925914413255890880747392740262672393793987046795024298262433148920536041259765624999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999992599320315969405975
#2: existing=13.9075498613921344064705338468
 LMFDB =13.90754986139213440644668132877
 diff=0.00000000000000000002385251803765112022760492797480211604451120171218385878318668513742295544943772256374359130859375000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000003223692095971801632
#3: existing=17.4427769782344733135480823313
 LMFDB =17.44277697823447331355152513713
 diff=0.000000000000000000003442805822469000698483411531472468785391851975199415140591052697516261105192825198173522949218749999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999848721

In [57]:
# Existing50 zeros match LMFDB to ~19 decimal places. Good.
# 
# Now let's commit to running the FULL 200-zero generation. Budget ~ 50 minutes.
# 
# First, also load LMFDB zeros at higher precision to ~30 digits.
# Then run my AFE-based finder to high precision.

# Set up the AFE function with sufficient bisection precision for matching 15+ decimal places.

# Re-import the module fresh
import importlib, sys
if '_zeros_v2' in sys.modules:
 del sys.modules['_zeros_v2']
import _zeros_v2 as kz2

# Validate first 10 zeros at HIGH precision and check against LMFDB
mp.mp.dps = 60
import time
t0 = time.time()
zeros_first10 = kz2.L_Delta_zeros(10, step=mp.mpf('0.3'), t_start=mp.mpf('8.0'),
 bisect_max=70, t_precision=mp.mpf('1e-30'),
 cache_suffix='afe_validation10',
 verbose=True)
print(f'First 10 zeros generated in {time.time()-t0:.1f}s')


 zero #1: γ = 9.2223793999211025222437246138965441510926268620096


 zero #2: γ = 13.907549861392134406446700274130080787760022076327


 zero #3: γ = 17.442776978234473313551570374381298467181267142223


 zero #4: γ = 19.65651314195496100012717004496074629793511689968


 zero #5: γ = 22.336103637209867275682739380378120930270569033383


 zero #6: γ = 25.274636548112365356745450965174175361749320245509


 zero #7: γ = 26.804391158350403032575859024549090603439793767393


 zero #8: γ = 28.831682624186875445021921245580993897750232690669


 zero #9: γ = 31.178209498360259064492071656260301082408403772207


 zero #10: γ = 32.774875382231207441830553786098374768886998253948
First 10 zeros generated in 193.1s


In [58]:
# 193s for first 10. Now compare to LMFDB:
print("Validation: AFE-computed zeros vs LMFDB:")
for i in range(10):
 afe = zeros_first10[i].imag
 lmfdb = mp.mpf(lmfdb_zeros_first10[i])
 diff = abs(afe - lmfdb)
 digits_match = -mp.log10(diff) if diff > 0 else 999
 print(f'#{i+1}: γ_afe={mp.nstr(afe, 25)}')
 print(f' γ_LMFDB={lmfdb_zeros_first10[i][:30]}')
 print(f' diff={mp.nstr(diff, 3)} ({float(digits_match):.1f} digits match)')


Validation: AFE-computed zeros vs LMFDB:
#1: γ_afe=9.222379399921102522243725
 γ_LMFDB=9.2223793999211025222437671927
 diff=4.26e-23 (22.4 digits match)
#2: γ_afe=13.9075498613921344064467
 γ_LMFDB=13.907549861392134406446681328
 diff=1.89e-23 (22.7 digits match)
#3: γ_afe=17.44277697823447331355157
 γ_LMFDB=17.442776978234473313551525137
 diff=4.52e-23 (22.3 digits match)
#4: γ_afe=19.65651314195496100012717
 γ_LMFDB=19.656513141954961000127281756
 diff=1.12e-22 (22.0 digits match)
#5: γ_afe=22.33610363720986727568274
 γ_LMFDB=22.336103637209867275682674459
 diff=6.49e-23 (22.2 digits match)
#6: γ_afe=25.27463654811236535674545
 γ_LMFDB=25.274636548112365356745324193
 diff=1.27e-22 (21.9 digits match)
#7: γ_afe=26.80439115835040303257586
 γ_LMFDB=26.804391158350403032575749233
 diff=1.1e-22 (22.0 digits match)
#8: γ_afe=28.83168262418687544502192
 γ_LMFDB=28.831682624186875445021961912
 diff=4.07e-23 (22.4 digits match)
#9: γ_afe=31.17820949836025906449207
 γ_LMFDB=31.1782094983602590

In [59]:
# EXCELLENT! All 10 LMFDB zeros match to 21-22 decimal places, FAR exceeding the 15-digit threshold 
# the task required.
# 
# Now generate the full 200 zeros. Start fresh (clear cache) and resume from existing50 file as 
# starting point to save time. Actually let me use my AFE-generated zeros as initial.

# Let me clear the cache and run fresh
import os
cache_dir = '/workspace/2fa3d8f6-4931-40d5-80bb-70f80c87743e/cache'
for f in os.listdir(cache_dir):
 if 'LDelta' in f:
 os.remove(os.path.join(cache_dir, f))
 print('Removed cache:', f)

# Use existing50 as starting initial_zeros for efficiency.
# But first: verify existing50 zeros are correct vs AFE by checking a few.
# We already verified the first 10 match exactly. Check the 50th:
mp.mp.dps = 80
v = kz2.Lambda_Delta(existing50[49].imag)
print(f'Λ at 50th existing zero (t={existing50[49].imag}): {v}')
# Should be very close to zero


Removed cache: LDelta_zeros_N10_afe_validation10.pkl


Λ at 50th existing zero (t=90.96347061642236287986136312594010015681625987174910816998410105016850679682249844547764): -8.466547299407262880889587377453033873944542125930914932619862449489302396883181350833559e-57


In [60]:
# Λ at the 50th existing zero is -8e-57, very close to zero. Good.
# 
# Strategy: run the full 200-zero generation using initial_zeros = existing50.
# Save progress every 5 zeros to track. Set verbose=True so I can monitor.
# 
# Given the cell runtime limits, I need to ensure the process completes within timeouts.
# Let me run it in chunks and persist.

# Plan: launch the generation. If it runs out of time, save what we have.
# Best to use a subagent for the long run, but let me first attempt directly with strict time management.

# Estimate: 150 more zeros to generate, dps growing from ~85 to ~220.
# Avg eval time: roughly average dps ~ 150 → about 3-4 s/eval.
# Avg evals per zero: scan + bisect ~ 3 + 70 = 73. But bisection at low precision can use 
# Brent's method or stop early.
# Per zero: 73 * 3 = 220s. 150 zeros: 33,000s = 9 hours. STILL TOO LONG.
# 
# Need MORE optimizations. Critical: bisect_max=70 is overkill. Match precision needed: 15 digits.
# Step is 0.3, so bisect_max=50 gives 0.3*2^(-50)=2.7e-16 — enough.
# Bisect_max=40 gives 2.7e-13. Borderline. Use 50.

# Per zero with bisect=50: 53 * 3 = 159s. 150 zeros = 6.6hr. Still too long.

# Real fix: Use Newton's method (mp.findroot 'newton') with derivative. 
# Or even better: TWO-STAGE approach. Stage 1: find bracket with low precision. Stage 2: refine 
# only with secant method (3-5 evals) at higher precision.

# Let me redesign with secant refinement and smaller scan step.

from _zeros_v2 import Lambda_Delta as Lambda_AFE, required_dps_for_t, _get_taus

def find_zeros_two_stage(N_target, step=0.3, t_start=8.0, n_taus=200, 
 initial_zeros=None, save_path=None, save_every=5, verbose=False):
 """Two-stage zero finder: scan + secant refinement."""
 taus = _get_taus(n_taus)
 zeros = list(initial_zeros) if initial_zeros else []
 if zeros:
 last_g = zeros[-1].imag if hasattr(zeros[-1], 'imag') else zeros[-1]
 t_start_curr = float(last_g) + 0.05
 else:
 t_start_curr = t_start
 
 t_prev = mp.mpf(t_start_curr)
 Lp = Lambda_AFE(t_prev, taus, n_taus)
 t = t_prev + mp.mpf(step)
 
 while len(zeros) < N_target:
 Lc = Lambda_AFE(t, taus, n_taus)
 if Lp * Lc < 0:
 # Refine via secant
 a_, b_ = t_prev, t
 La, Lb = Lp, Lc
 # Standard secant with safeguard
 try:
 # secant iterations
 for _ in range(20):
 if Lb == La:
 break
 m_ = b_ - Lb * (b_ - a_) / (Lb - La)
 # Ensure m_ stays in [a_, b_]
 a_low, b_high = min(a_, b_), max(a_, b_)
 if m_ < a_low or m_ > b_high:
 m_ = (a_low + b_high) / 2
 Lm = Lambda_AFE(m_, taus, n_taus)
 if abs(Lm) < mp.mpf(10)**(-mp.mp.dps + 5):
 a_ = b_ = m_
 break
 # Update bracket
 if La * Lm < 0:
 b_, Lb = m_, Lm
 else:
 a_, La = m_, Lm
 gamma = (a_ + b_) / 2 if a_ != b_ else m_
 zeros.append(mp.mpc(mp.mpf('0.5'), gamma))
 if verbose:
 print(f' zero #{len(zeros)}: γ = {gamma}', flush=True)
 if save_path and len(zeros) % save_every == 0:
 with open(save_path, 'wb') as f:
 pickle.dump(zeros, f)
 except Exception as e:
 print(f'Failed refinement: {e}')
 t_prev, Lp = t, Lc
 t += mp.mpf(step)
 return zeros

# Test on first 5 zeros
import time
mp.mp.dps = 50
t0 = time.time()
zz_test = find_zeros_two_stage(5, step=0.3, t_start=8.0, verbose=True)
print(f'5 zeros in {time.time()-t0:.1f}s')


 zero #1: γ = 9.2111896999605512389174231038686082773646366900201


 zero #2: γ = 13.803774930696068318914782732489896402129527421529


 zero #3: γ = 17.371388489117236575581100903137565401849648339974


 zero #4: γ = 19.528256570977484578047787415558388187619021001965


 zero #5: γ = 22.218051818604946639312380277772811441310886753326


5 zeros in 30.7s


In [61]:
# PROBLEM! The values are wrong! γ_1 = 9.2111 not 9.2222. 
# Why? My secant code has a bug — `m_` outside bracket gets midpoint, then I use `gamma = (a_+b_)/2`
# but a_ and b_ are updated. Let me debug.

# Actually probably the issue: after secant updates, La can have wrong sign relationship.
# Let me look: I set La*Lm < 0 → update b. But La might have become 0 or wrong. Let me use a 
# cleaner Brent's method or just go back to bisection with mp.findroot wrapper.

# Use mpmath findroot with bracket + 'anderson' or 'illinois':
import time
mp.mp.dps = 80
t0 = time.time()

t_prev = mp.mpf(8.0)
Lp = Lambda_AFE(t_prev, taus200, 200)
t = t_prev + mp.mpf('0.3')
zeros_test = []
while len(zeros_test) < 5:
 Lc = Lambda_AFE(t, taus200, 200)
 if Lp * Lc < 0:
 # Use bisection for reliability:
 a_, b_ = t_prev, t
 La = Lp
 for _ in range(60):
 if b_-a_ < mp.mpf('1e-30'):
 break
 m_ = (a_+b_)/2
 Lm = Lambda_AFE(m_, taus200, 200)
 if Lm == 0:
 a_=b_=m_; break
 if La*Lm < 0:
 b_ = m_
 else:
 a_, La = m_, Lm
 zeros_test.append((a_+b_)/2)
 t_prev, Lp = t, Lc
 t += mp.mpf('0.3')

for z in zeros_test:
 print(z)
print(f'Time: {time.time()-t0:.1f}s')


9.2223793999211025223302490294585709307284560054541
13.907549861392134406571849392086903662857366725803
17.442776978234473313515359715886177127686096355319
19.656513141954961000146609451100232490716734901071
22.336103637209867275740041159259874348208541050554
Time: 77.4s


In [62]:
# OK bisection works correctly. 77s for 5 zeros at low t. That gives ~15s/zero in early regime.
# 
# For 200 zeros at avg 15-60s = 3000-12000s. Maybe acceptable.
# 
# Let me commit to: bisection, 50 iter max, t_precision=1e-20 (16 digits past leading), and run.
# Use SAVE checkpointing in case of timeout.
# 
# Also OPTIMIZATION: since bisection at low t uses dps=50 (cheap) but bisection at high t uses 
# dps=200 (~10s each), at high t I want to minimize bisection iters by using a tighter bracket 
# starting point (smaller scan step).
# 
# Or BETTER: do scan once with COARSE step but compute Lambda only at coarse points. When sign 
# change found, switch to BINARY SEARCH but with a CACHE of intermediate Lambda values so we don't 
# recompute.
# 
# Actually the bisection IS the most expensive part. ~50 iter × eval at high t (10s) = 500s per zero
# at the very end. 50 zeros at this rate ~ 7 hours. Way too much.
# 
# RADICAL FIX: Reduce precision requirement. The task says match LMFDB "to at least 15 decimal places".
# We only have LMFDB to 28 digits. So we need 15 digits precision in the COMPUTED zero. 
# 
# Bisection from 0.3-wide bracket needs ceil(log2(0.3/1e-15)) = 49 iterations. That's what we have.
# Can't easily reduce without losing accuracy.
# 
# But — at HIGH t, the EVALUATION of Lambda already costs a lot, but most of those iters could be at 
# LOWER DPS once we're well-bracketed.
# 
# Two-stage scheme: 
# Stage A (bracket finding): use dps_lower until bracket is tight (e.g. 1e-15)
# Stage B (final refinement): increase dps if needed for ultra-high precision
# 
# Actually wait — at high t, our scan step is 0.3, and the SIGN at coarse points uses high dps. 
# Once bracketed in 0.3-window, each bisection halves the interval. The TRUE zero is somewhere in 
# this interval. The sign of Λ at intermediate point doesn't fundamentally need dps higher than 
# what gives reliable sign there. 
# 
# But our dps formula sets dps based on t, not on |Λ|. At a midpoint we still have similar |Λ| 
# magnitude as at the endpoints (away from the zero itself). So dps should be similar.
# 
# OK let me just commit and run. Time budget says 3600s for this task. Use ~ 50 min.

# Final plan:
# 1. Load existing50 as initial.
# 2. Run from zero #51 onward with progressive checkpointing.
# 3. Save every 5 zeros so we can recover.
# 4. After ~45 minutes, stop and submit whatever we have.

# First clear cache
import os
cache_dir = '/workspace/2fa3d8f6-4931-40d5-80bb-70f80c87743e/cache'
os.makedirs(cache_dir, exist_ok=True)
for f in os.listdir(cache_dir):
 if 'LDelta' in f:
 os.remove(os.path.join(cache_dir, f))

# Set up: use existing50 as initial. 
initial_zeros = existing50.copy()
print(f'Starting from {len(initial_zeros)} existing zeros, last γ = {initial_zeros[-1].imag}')

# Save the initial zeros immediately for safety
final_path = '/workspace/2fa3d8f6-4931-40d5-80bb-70f80c87743e/L_Delta_zeros_200_reliable.pkl'
with open(final_path, 'wb') as f:
 pickle.dump(initial_zeros, f)
print(f'Saved {len(initial_zeros)} initial zeros to {final_path}')


Starting from 50 existing zeros, last γ = 90.963470616422362879861363125940100156816259871749
Saved 50 initial zeros to /workspace/2fa3d8f6-4931-40d5-80bb-70f80c87743e/L_Delta_zeros_200_reliable.pkl


In [63]:
# Plan: launch zero generation. Save EVERY zero (not every 5) so if interrupted we can resume.
# Use bisect_max=50 and t_precision=1e-20 — gives 15-20 digit precision.

import time
import signal

class TimeoutException(Exception):
 pass

def run_zero_generation_with_budget(target=200, time_budget_s=2700, save_path=final_path):
 """Generate zeros up to `target`, saving incrementally. Stops at time budget."""
 # Load whatever's saved
 with open(save_path, 'rb') as f:
 zeros = pickle.load(f)
 print(f'Resuming from {len(zeros)} zeros (last γ = {zeros[-1].imag if zeros else "N/A"})', flush=True)
 
 if len(zeros) >= target:
 print(f'Already have {len(zeros)} >= target {target}')
 return zeros
 
 taus = _get_taus(200)
 step = mp.mpf('0.3')
 t_start = mp.mpf(zeros[-1].imag) + mp.mpf('0.1') if zeros else mp.mpf('8.0')
 
 t_prev = t_start
 Lp = Lambda_AFE(t_prev, taus, 200)
 t = t_prev + step
 
 start_time = time.time()
 last_save_time = start_time
 
 while len(zeros) < target:
 if time.time() - start_time > time_budget_s:
 print(f'Time budget exceeded ({time.time()-start_time:.0f}s). Stopping at {len(zeros)} zeros.', flush=True)
 break
 
 Lc = Lambda_AFE(t, taus, 200)
 if Lp * Lc < 0:
 # Bisection refinement
 a_, b_ = t_prev, t
 La = Lp
 for it_b in range(50):
 if b_ - a_ < mp.mpf('1e-20'):
 break
 m_ = (a_+b_)/2
 Lm = Lambda_AFE(m_, taus, 200)
 if Lm == 0:
 a_=b_=m_; break
 if La*Lm < 0:
 b_ = m_
 else:
 a_, La = m_, Lm
 gamma = (a_+b_)/2
 zeros.append(mp.mpc(mp.mpf('0.5'), gamma))
 elapsed = time.time() - start_time
 print(f' zero #{len(zeros)}: γ = {mp.nstr(gamma, 20)} ({elapsed:.1f}s elapsed, dps={mp.mp.dps})', flush=True)
 # Save every zero
 with open(save_path, 'wb') as f:
 pickle.dump(zeros, f)
 t_prev, Lp = t, Lc
 t += step
 
 return zeros

# Test with a small budget first to verify behavior
zeros_so_far = run_zero_generation_with_budget(target=55, time_budget_s=600, save_path=final_path)
print(f'Final count: {len(zeros_so_far)}')


Resuming from 50 zeros (last γ = 90.963470616422362879861363125940100156816259871749)


 zero #51: γ = 91.112718531472495298 (70.4s elapsed, dps=88)


 zero #52: γ = 92.442925494721278257 (145.4s elapsed, dps=89)


 zero #53: γ = 93.769123947436761435 (223.8s elapsed, dps=90)


 zero #54: γ = 95.138078539773485376 (304.5s elapsed, dps=91)


 zero #55: γ = 95.624921077045154882 (383.1s elapsed, dps=91)


Final count: 55


In [64]:
# 5 zeros in 383s = ~77s/zero at t~90-95. Plus dps=88-91. Scaling:
# At t=200, dps=165, ~ 200s/zero estimate.
# At t=270, dps=214, ~ 350s/zero estimate.
# 
# So to do 145 more zeros: avg ~ 200s = 29000s = 8 hours. NOT FEASIBLE in single task.
# 
# OPTIMIZATION REQUIRED. Let me think about what's eating time.
# 
# 50 bisection iterations × Lambda_AFE eval. At dps=90, AFE takes ~1.5s. 50*1.5 = 75s per zero. 
# That's the bisection cost. 
# 
# Optimization 1: Stop bisecting once precision is enough. Bisect to 1e-15, that's 49 iter for 0.3 
# range. We can stop earlier — once bracket width < 1e-15.
# 
# Optimization 2: Use a smarter algorithm — secant/Brent. Converges in ~10 iter instead of 50.
# 
# Optimization 3: After scan finds bracket, use mp.findroot('mnewton') with derivative.
# 
# Let me use mpmath's mp.findroot with secant. It's robust if given good bracket.

mp.mp.dps = 90
import time

def Lambda_fn(t):
 return Lambda_AFE(t, taus200, 200)

# Test with anderson
t0 = time.time()
root = mp.findroot(Lambda_fn, [mp.mpf('91.0'), mp.mpf('91.3')], solver='anderson')
print(f'Anderson: {root} ({time.time()-t0:.2f}s)')

t0 = time.time()
root = mp.findroot(Lambda_fn, mp.mpf('91.15'), solver='secant')
print(f'Secant: {root} ({time.time()-t0:.2f}s)')

t0 = time.time()
root = mp.findroot(Lambda_fn, [mp.mpf('91.0'), mp.mpf('91.3')], solver='illinois')
print(f'Illinois: {root} ({time.time()-t0:.2f}s)')


Anderson: 91.1127185314724952421256068998767289017410081873361894992191032998547957271945271029934025 (14.99s)


Secant: 91.1127185314724952421256068998767289017435546906413097227996673780568089970181813345106674 (48.79s)


Illinois: 91.1127185314724952421256068998767289017420447878829623266094407700469673692435391244716633 (17.76s)


In [65]:
# Anderson and Illinois methods are faster (~15s) and converge to same value as bisection 
# (~76s). 
# 
# Speedup factor: ~5x. So at t=90 we go from 75s/zero to 15s/zero. At t=270, from 350s to 70s.
# Avg over 150 remaining zeros: ~40s. Total: 6000s = 100min. Still too much!
# 
# But! 'illinois' should be very fast. Let me confirm by trying tighter brackets and looser tolerance.

mp.mp.dps = 90 
t0 = time.time()
root = mp.findroot(Lambda_fn, [mp.mpf('91.0'), mp.mpf('91.3')], solver='illinois', tol=mp.mpf('1e-15'))
print(f'Illinois tol=1e-15: {root} ({time.time()-t0:.2f}s)')

# Compare also at higher t
mp.mp.dps = 150
t0 = time.time()
root = mp.findroot(Lambda_fn, [mp.mpf('150'), mp.mpf('150.3')], solver='illinois', tol=mp.mpf('1e-15'))
print(f'Illinois t=150: {root} ({time.time()-t0:.2f}s)')


Illinois tol=1e-15: 91.1248722095218893807033164332014076173010788728391478627030173890035629453267940451646655 (6.03s)


Illinois t=150: 150.286165951065993553635367349854296767118495653547838911764582203134776660150276429654777744010409809821369769489003078061066299364339707092975070833 (21.08s)


In [66]:
# Time pressure. Let me move fast and accept we won't get 200 zeros, but we will fully validate
# the corrected method and produce as many zeros as time allows.
# 
# Wait — illinois at 91.0-91.3 may not have given a true zero! 91.124 is NOT in [91.0, 91.3]'s
# zero range. We've already established zero 51 is at 91.112. So illinois found something different.
# Actually 91.1248 != 91.1127. Hmm.
# 
# Wait — let me check; bisection gave 91.112718. Illinois gave 91.124872. Different!
# Maybe the bracket [91.0, 91.3] contains TWO zeros? Or maybe illinois converged to a different sign change.
# 
# Let me check Lambda at a few points:
for t_test in [91.0, 91.1, 91.11, 91.12, 91.125, 91.13, 91.15, 91.2, 91.25, 91.3]:
 v = Lambda_AFE(mp.mpf(t_test), taus200, 200)
 print(f't={t_test}: Λ={float(v):+.3e}')


t=91.0: Λ=-6.423e-57


t=91.1: Λ=-7.111e-58


t=91.11: Λ=-1.514e-58


t=91.12: Λ=+4.036e-58


t=91.125: Λ=+6.790e-58


t=91.13: Λ=+9.531e-58


t=91.15: Λ=+2.034e-57


t=91.2: Λ=+4.604e-57


t=91.25: Λ=+6.942e-57


t=91.3: Λ=+9.008e-57


In [67]:
# OK there's only ONE zero in [91.0, 91.3], at t ~ 91.112-91.116. Bisection gave 91.1127, illinois 
# gave 91.1248. Illinois got it WRONG. Maybe its tolerance was bad or it diverged.
# 
# Actually wait — 91.1127 IS in [91.11, 91.12] per signs above. Illinois at 91.1248 is way off.
# Maybe my Lambda_fn used different dps when called from findroot than from bisection.
# 
# Given the time pressure, let me commit to running with what we know works (bisection with the
# corrected AFE function) for as many zeros as time permits, and save and submit.
# 
# Currently we have 55 zeros saved. Let me launch another 10-15 minutes of generation.

# I have ~1170s. Let me budget 900s for further generation, leaving 270s for finalization.
import time
zeros_so_far = run_zero_generation_with_budget(target=200, time_budget_s=850, save_path=final_path)
print(f'Final zero count: {len(zeros_so_far)}')
print(f'Last zero: γ = {zeros_so_far[-1].imag}')


TimeoutError: Code execution timed out after 563.0 seconds